# LeetCode 282 — Expression Add Operators (Hard) bằng Python

Notebook này giảng nhập môn bài toán:

```python
class Solution:
    def addOperators(self, num: str, target: int) -> List[str]:

```

**Mục tiêu sau khi học xong:**

1. Hiểu đề: chèn `+`, `-`, `*` vào **giữa** các chữ số của `num` (hoặc **không chèn gì** để **nối** các chữ số thành số nhiều chữ số) sao cho biểu thức có giá trị bằng `target`, và trả về **tất cả** biểu thức như vậy.
2. Nắm chắc 3 "luật chơi" nền tảng trước khi viết thuật toán:
   - **ghép chữ số thành số**: đọc `"105"` từ trái sang phải được `1 → 10 → 105` nhờ công thức `cur * 10 + d`;
   - **không có số 0 đứng đầu** (leading zero): `"05"` là toán hạng **không hợp lệ**, còn `"0"` thì **hợp lệ**;
   - **độ ưu tiên toán tử**: `*` được tính **trước** `+` và `-`, nên `1+2*3 = 7` chứ không phải `9`.
3. Viết **Cách 1 — brute force**: dùng `itertools.product` sinh mọi cách đặt 4 lựa chọn (`''`, `'+'`, `'-'`, `'*'`) vào các khe giữa hai chữ số, lọc số 0 đứng đầu rồi `eval`. Cách này chậm (`4^(n-1)` biểu thức) nhưng **dễ tin**, nên ta dùng nó làm **trọng tài** để kiểm tra các cách sau.
4. Học **đệ quy / backtracking (quay lui)** từ con số 0 và viết **Cách 2 — backtracking xây chuỗi biểu thức rồi `eval` ở cuối**.
5. Hiểu **mấu chốt của bài**: vì sao chỉ giữ một biến `value` là chưa đủ khi có phép nhân, và vì sao giữ thêm **`prev` — giá trị hạng tử cuối** cho phép cập nhật phép nhân bằng công thức `value - prev + prev * cur`. Đây là **Cách 3 — cách chính để nộp**.
6. **Trace cây gọi** đệ quy cho `"123", 6` và `"105", 5` để nhìn thấy `value`, `prev` thay đổi và thấy nhánh số 0 đứng đầu bị cắt.
7. Biết **Cách 4 (tùy chọn)**: giữ `path` dạng list với `append`/`pop` thay vì nối chuỗi.
8. Chứng minh vì sao thuật toán đúng, phân tích Big-O `O(n · 4^n)`, các lỗi người mới hay gặp, bài liên quan và **6 bài tập** có gợi ý + đáp án.

## 1. Đề bài

Cho một chuỗi `num` chỉ gồm các chữ số và một số nguyên `target`. Hãy trả về **tất cả các khả năng** chèn các toán tử hai ngôi (binary operator) `'+'`, `'-'` và/hoặc `'*'` vào **giữa các chữ số** của `num` sao cho biểu thức thu được có giá trị bằng `target`.

**Lưu ý 1:** các toán hạng (operand — các số đứng giữa những toán tử) trong biểu thức trả về **không được có số 0 đứng đầu** (leading zeros).

**Lưu ý 2:** một toán hạng có thể gồm **nhiều chữ số** (ví dụ `"10-5"` dùng toán hạng `10`).

**Ví dụ 1**

```text
Input:  num = "123", target = 6
Output: ["1*2*3","1+2+3"]
Explanation: Both "1*2*3" and "1+2+3" evaluate to 6.
```

**Ví dụ 2**

```text
Input:  num = "232", target = 8
Output: ["2*3+2","2+3*2"]
Explanation: Both "2*3+2" and "2+3*2" evaluate to 8.
```

**Ví dụ 3**

```text
Input:  num = "3456237490", target = 9191
Output: []
Explanation: There are no expressions that can be created from "3456237490" to evaluate to 9191.
```

**Constraints (ràng buộc):**

- `1 <= num.length <= 10`
- `num` chỉ gồm các chữ số.
- `-2^31 <= target <= 2^31 - 1`

Ngoài 3 ví dụ trên, notebook dùng thêm 2 ví dụ kinh điển của bài này (có trong các phiên bản đề cũ) vì chúng chạm đúng luật số 0:

```text
num = "105", target = 5  ->  ["1*0+5","10-5"]
num = "00",  target = 0  ->  ["0+0","0-0","0*0"]
```

### Đề nói "trả về tất cả" — thứ tự có quan trọng không?

LeetCode chấp nhận đáp án theo **thứ tự bất kỳ**. Vì thế khi tự test, ta **không** so sánh hai list trực tiếp (`["a", "b"] != ["b", "a"]`), mà so sánh sau khi **sắp xếp** cả hai bằng `sorted(...)`. Cell dưới lưu các ví dụ và viết sẵn hàm `same_answer` để dùng trong cả notebook.

In [1]:
from typing import List

examples = [
    ("123", 6, ["1*2*3", "1+2+3"]),
    ("232", 8, ["2*3+2", "2+3*2"]),
    ("3456237490", 9191, []),
    ("105", 5, ["1*0+5", "10-5"]),     # ví dụ kinh điển: số 0 ở giữa
    ("00", 0, ["0+0", "0-0", "0*0"]),  # ví dụ kinh điển: "00" không hợp lệ
]

for num, target, expected in examples:
    print(f"num = {num!r:<13} target = {target:<5} expected = {expected}")


def same_answer(actual: List[str], expected: List[str]) -> bool:
    # So sánh 2 list biểu thức, bỏ qua thứ tự
    return sorted(actual) == sorted(expected)

print(["1+2+3", "1*2*3"] == ["1*2*3", "1+2+3"])             # False: khác thứ tự
print(same_answer(["1+2+3", "1*2*3"], ["1*2*3", "1+2+3"]))  # True

num = '123'         target = 6     expected = ['1*2*3', '1+2+3']
num = '232'         target = 8     expected = ['2*3+2', '2+3*2']
num = '3456237490'  target = 9191  expected = []
num = '105'         target = 5     expected = ['1*0+5', '10-5']
num = '00'          target = 0     expected = ['0+0', '0-0', '0*0']
False
True


### 1.1. Hiểu đề bằng tay: "khe" giữa hai chữ số

Viết `num = "123"` thành `1 _ 2 _ 3`. Mỗi dấu `_` là một **khe** (gap) giữa hai chữ số liền nhau. Ở **mỗi khe** ta có đúng **4 lựa chọn**:

| Lựa chọn ở khe | Ý nghĩa | Ví dụ với khe giữa `1` và `2` |
|---|---|---|
| `''` (để trống) | **nối** hai chữ số thành một số dài hơn | `12...` |
| `'+'` | cộng | `1+2...` |
| `'-'` | trừ | `1-2...` |
| `'*'` | nhân | `1*2...` |

Với `"123"` có 2 khe nên có `4 × 4 = 16` biểu thức. Một vài biểu thức tiêu biểu:

| Khe 1 | Khe 2 | Biểu thức | Giá trị | Ghi chú |
|:---:|:---:|---|---:|---|
| nối | nối | `123` | 123 | không chèn toán tử nào cũng là một biểu thức hợp lệ |
| `+` | `+` | `1+2+3` | 6 | ✔ bằng target 6 |
| `*` | `*` | `1*2*3` | 6 | ✔ bằng target 6 |
| `+` | `*` | `1+2*3` | 7 | nhân trước: `1 + 6` |
| nối | `-` | `12-3` | 9 | toán hạng `12` có 2 chữ số |
| `-` | nối | `1-23` | -22 | toán hạng `23` có 2 chữ số |

**Những điều đề KHÔNG cho phép:**

- Không được đặt toán tử **trước chữ số đầu tiên** (như `-1+2+3`): toán tử là **hai ngôi** — cần một toán hạng bên trái và một bên phải — nên không có "dấu trừ đứng đầu".
- Không được đổi thứ tự chữ số, không được bỏ chữ số nào, không được thêm dấu ngoặc.

**Tổng quát:** `n` chữ số có `n - 1` khe, mỗi khe 4 lựa chọn ⇒ có `4^(n-1)` biểu thức (trước khi loại các biểu thức có số 0 đứng đầu). Với `n = 10` là `4^9 = 262144` biểu thức — đây là con số ta sẽ gặp lại khi phân tích độ phức tạp.

In [2]:
# Kiểm tra bảng trên bằng eval (eval sẽ được giải thích kỹ ở mục 2.4)
for expr in ["123", "1+2+3", "1*2*3", "1+2*3", "12-3", "1-23"]:
    print(f"{expr:<6} = {eval(expr)}")

# Số biểu thức 4^(n-1) theo số chữ số n
for n in [1, 2, 3, 5, 10]:
    print(f"n = {n:>2} chữ số -> {n - 1} khe -> 4^{n - 1} = {4 ** (n - 1)} biểu thức")

123    = 123
1+2+3  = 6
1*2*3  = 6
1+2*3  = 7
12-3   = 9
1-23   = -22
n =  1 chữ số -> 0 khe -> 4^0 = 1 biểu thức
n =  2 chữ số -> 1 khe -> 4^1 = 4 biểu thức
n =  3 chữ số -> 2 khe -> 4^2 = 16 biểu thức
n =  5 chữ số -> 4 khe -> 4^4 = 256 biểu thức
n = 10 chữ số -> 9 khe -> 4^9 = 262144 biểu thức


## 2. Kiến thức nền cho bài này

### 2.1. Ghép chữ số thành số nhiều chữ số

Khi "nối" chữ số, ta đang tạo ra một **toán hạng** dài hơn. Đọc từ trái sang phải, mỗi lần thêm chữ số `d` vào **đuôi** số `cur` hiện có thì:

```python
cur = cur * 10 + d
```

Vì sao nhân 10? Vì thêm một chữ số vào cuối làm mọi chữ số cũ **dịch sang trái một hàng** (hàng đơn vị → hàng chục → hàng trăm), rồi chữ số mới đứng ở hàng đơn vị.

Ví dụ đọc `"105"`:

```text
cur = 0
đọc '1': cur = 0 * 10 + 1  = 1
đọc '0': cur = 1 * 10 + 0  = 10
đọc '5': cur = 10 * 10 + 5 = 105
```

Cách thứ hai (dễ đọc hơn) là **cắt chuỗi** rồi đổi sang số: `int(num[i:j + 1])`. Nhắc lại cú pháp cắt chuỗi: `s[a:b]` lấy các ký tự từ chỉ số `a` **đến trước** `b`. Vậy `num[i:j + 1]` là đoạn từ `i` đến **hết** `j`.

Trong thuật toán, tại vị trí `i` ta sẽ thử **mọi toán hạng bắt đầu tại `i`**: `num[i:i+1]`, `num[i:i+2]`, … — mỗi cái dài hơn cái trước đúng một chữ số.

In [3]:
num = "105"

# Cách 1: cur * 10 + d
cur = 0
for ch in num:
    d = int(ch)              # '5' -> 5
    cur = cur * 10 + d
    print(f"đọc {ch!r}: cur = {cur}")

# Cách 2: cắt chuỗi rồi int(...)
print(int(num[0:3]), int(num[0:2]), int(num[0:1]))   # 105 10 1

# Mọi toán hạng bắt đầu tại i = 0 của "105"
i = 0
for j in range(i, len(num)):
    print(f"num[{i}:{j + 1}] = {num[i:j + 1]!r}")

đọc '1': cur = 1
đọc '0': cur = 10
đọc '5': cur = 105
105 10 1
num[0:1] = '1'
num[0:2] = '10'
num[0:3] = '105'


### 2.2. Luật "không có số 0 đứng đầu" (no leading zeros)

Một toán hạng được coi là **không hợp lệ** nếu nó có **từ 2 chữ số trở lên** và **bắt đầu bằng `'0'`**:

| Toán hạng | Hợp lệ? | Vì sao |
|---|:---:|---|
| `"0"` | ✔ | chỉ một chữ số `0` — đây là số 0 bình thường |
| `"5"`, `"10"`, `"105"` | ✔ | số 0 ở giữa/cuối không sao |
| `"05"` | ✘ | dài 2 và bắt đầu bằng `0` |
| `"00"` | ✘ | dài 2 và bắt đầu bằng `0` (dù giá trị là 0!) |
| `"012"` | ✘ | dài 3 và bắt đầu bằng `0` |

Điều kiện bằng code: `len(s) > 1 and s[0] == "0"` ⇒ **không hợp lệ**.

**Hệ quả rất quan trọng cho thuật toán:** nếu toán hạng bắt đầu tại vị trí có chữ số `'0'` thì chỉ có **đúng một** lựa chọn là `"0"`. Mọi toán hạng dài hơn (`"05"`, `"050"`, …) đều bắt đầu bằng `'0'` nên **đều hỏng**. Vì vậy khi thử dần các toán hạng dài ra, gặp toán hạng đầu tiên bị hỏng thì có thể **dừng hẳn (`break`)**, không cần thử tiếp.

**Cái bẫy:** `int("05")` trả về `5` một cách "êm ru" — Python không báo lỗi. Nếu ta đổi sang số trước rồi mới kiểm, thông tin "có số 0 đứng đầu" đã **mất**. Luôn kiểm trên **chuỗi**.

In [4]:
def is_valid_operand(s: str) -> bool:
    # Hợp lệ nếu chỉ có 1 chữ số, hoặc không bắt đầu bằng '0'
    return len(s) == 1 or s[0] != "0"

for s in ["0", "5", "10", "105", "05", "00", "012"]:
    print(f"{s!r:<6} hợp lệ? {is_valid_operand(s)}")

print("int('05') =", int("05"), " <- int() che mất số 0 đứng đầu!")

# Các toán hạng bắt đầu tại i = 1 của "1005": gặp '0' thì chỉ "0" là hợp lệ
num, i = "1005", 1
for j in range(i, len(num)):
    s = num[i:j + 1]
    print(f"num[{i}:{j + 1}] = {s!r:<6} hợp lệ? {is_valid_operand(s)}")

'0'    hợp lệ? True
'5'    hợp lệ? True
'10'   hợp lệ? True
'105'  hợp lệ? True
'05'   hợp lệ? False
'00'   hợp lệ? False
'012'  hợp lệ? False
int('05') = 5  <- int() che mất số 0 đứng đầu!
num[1:2] = '0'    hợp lệ? True
num[1:3] = '00'   hợp lệ? False
num[1:4] = '005'  hợp lệ? False


### 2.3. Độ ưu tiên toán tử và khái niệm "hạng tử" (term)

Trong toán học (và trong Python), phép `*` được tính **trước** `+` và `-`. Còn `+` và `-` cùng mức thì tính **từ trái sang phải**.

| Biểu thức | Cách tính đúng | Giá trị |
|---|---|---:|
| `1+2*3` | `1 + (2*3)` | 7 |
| `2-3*2` | `2 - (3*2)` | -4 |
| `1+2*3*4` | `1 + (2*3*4)` | 25 |
| `10-5-2` | `(10-5) - 2` | 3 |
| `12-3*4+5` | `12 - (3*4) + 5` | 5 |

Một cách nhìn giúp ích rất nhiều cho bài này: **cắt biểu thức tại các dấu `+` và `-`**. Mỗi mảnh thu được là một **hạng tử** (term) — một **tích** các số nối với nhau bằng `*` — kèm theo **dấu** đứng trước nó. Giá trị biểu thức = **tổng các hạng tử có dấu**.

Ví dụ `12-3*4+5`:

```text
hạng tử 1: +12
hạng tử 2: -(3*4) = -12
hạng tử 3: +5
tổng       = 12 - 12 + 5 = 5
```

Ví dụ `1+2*3*4`: hạng tử là `+1` và `+(2*3*4) = +24`, tổng `25`.

Ghi nhớ: phép `*` **chỉ "dính" vào hạng tử đứng ngay trước nó**, không đụng tới các hạng tử khác. Đây chính là chìa khóa của lời giải chính ở mục 6.

In [5]:
for expr in ["1+2*3", "(1+2)*3", "2-3*2", "1+2*3*4", "10-5-2", "12-3*4+5", "2*3+2", "2+3*2"]:
    print(f"{expr:<9} = {eval(expr)}")

# Kiểm tra "tổng các hạng tử có dấu" cho 12-3*4+5
terms = [+12, -(3 * 4), +5]
print("các hạng tử:", terms, "-> tổng =", sum(terms))

1+2*3     = 7
(1+2)*3   = 9
2-3*2     = -4
1+2*3*4   = 25
10-5-2    = 3
12-3*4+5  = 5
2*3+2     = 8
2+3*2     = 8
các hạng tử: [12, -12, 5] -> tổng = 5


### 2.4. `eval` là gì, và vì sao chỉ dùng nó làm "trọng tài"

`eval(s)` nhận một **chuỗi** chứa biểu thức Python và trả về **giá trị** của nó, ví dụ `eval("1+2*3")` cho `7`. Python tự lo độ ưu tiên toán tử, nên `eval` là cách **dễ tin nhất** để biết một biểu thức bằng bao nhiêu.

Nhưng có 3 lý do không dùng `eval` trong lời giải chính:

1. **Chậm:** mỗi lần gọi, Python phải **phân tích cú pháp** (parse) cả chuỗi rồi mới tính. Gọi hàng trăm nghìn lần là rất tốn.
2. **Không an toàn** với dữ liệu lạ: `eval` chạy được **mọi** code Python, nên trong phần mềm thật không bao giờ `eval` chuỗi người dùng gửi lên. (Ở đây chuỗi do chính ta tạo nên không sao.)
3. **Không đúng luật số 0 của đề:** Python 3 **báo lỗi** `SyntaxError` với `"05"` nhưng lại **chấp nhận** `"00"` (vì `00` là cách viết hợp lệ của số 0 trong Python). Luật của đề thì loại **cả hai**. Vậy ta vẫn phải **tự** lọc số 0 đứng đầu trước khi `eval`.

Kết luận: `eval` rất hợp để viết **lời giải trọng tài** (chậm nhưng chắc), dùng để **so sánh** với các lời giải nhanh hơn.

In [6]:
print(eval("1+2*3"))      # 7
print(eval("1+00"))       # 1   <- Python chấp nhận "00", nhưng đề thì KHÔNG

try:
    eval("1*05")
except SyntaxError as e:
    print("eval('1*05') lỗi SyntaxError:", e.msg)

7
1
eval('1*05') lỗi SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers


### 2.5. `itertools.product` — sinh mọi tổ hợp lựa chọn, và `zip` để ghép

**`itertools.product(choices, repeat=k)`** sinh **tích Descartes** (Cartesian product): mọi bộ `k` phần tử, mỗi phần tử lấy từ `choices`. Nó giống như `k` vòng `for` lồng nhau mà không phải viết tay từng vòng.

```python
product(["+", "-"], repeat=2)
# ('+', '+'), ('+', '-'), ('-', '+'), ('-', '-')     -> 2^2 = 4 bộ
```

Với bài này: `choices = ["", "+", "-", "*"]` và `repeat = n - 1` (số khe) ⇒ ra đúng `4^(n-1)` bộ. Mỗi bộ `ops` cho biết khe thứ `k` đặt gì.

Chú ý trường hợp biên: khi `n = 1` thì `repeat = 0`, và `product(..., repeat=0)` sinh ra **đúng một** bộ rỗng `()` — tức là "không có khe nào, biểu thức chỉ là chính chữ số đó". Rất tiện: không cần xử lý riêng `n = 1`.

**`zip(a, b)`** đi song song hai dãy: lấy cặp `(a[0], b[0])`, `(a[1], b[1])`, … Ta dùng nó để ghép: bắt đầu bằng chữ số đầu `num[0]`, rồi với mỗi cặp (chữ số tiếp theo, toán tử của khe trước nó) thì nối `op + digit`.

In [7]:
from itertools import product

print(list(product(["+", "-"], repeat=2)))

OPS = ["", "+", "-", "*"]          # "" nghĩa là nối hai chữ số
all_ops = list(product(OPS, repeat=2))
print("số bộ với 2 khe:", len(all_ops))
print("6 bộ đầu:", all_ops[:6])
print("repeat=0 ->", list(product(OPS, repeat=0)))   # [()] : đúng 1 bộ rỗng

num, ops = "123", ("+", "*")
print(list(zip(num[1:], ops)))      # [('2', '+'), ('3', '*')]

def build_expression(num: str, ops) -> str:
    expr = num[0]
    for digit, op in zip(num[1:], ops):
        expr += op + digit          # op của khe đứng TRƯỚC digit
    return expr

print(build_expression("123", ("+", "*")))   # 1+2*3
print(build_expression("123", ("", "-")))    # 12-3
print(build_expression("7", ()))             # 7

[('+', '+'), ('+', '-'), ('-', '+'), ('-', '-')]
số bộ với 2 khe: 16
6 bộ đầu: [('', ''), ('', '+'), ('', '-'), ('', '*'), ('+', ''), ('+', '+')]
repeat=0 -> [()]
[('2', '+'), ('3', '*')]
1+2*3
12-3
7


## 3. Cách 1 — Brute force: sinh mọi biểu thức bằng `product`, lọc, rồi `eval`

Đây là cách người mới thường nghĩ ra đầu tiên, và nó bám **sát từng chữ của đề**:

1. Với mỗi bộ `ops` trong `product(OPS, repeat=n-1)`: dựng biểu thức `expr` bằng `build_expression`.
2. Tách `expr` ra các **toán hạng**; nếu có toán hạng nào có số 0 đứng đầu thì **bỏ qua** biểu thức này.
3. Còn lại thì `eval(expr)`; nếu bằng `target` thì thêm vào kết quả.

**Tách toán hạng thế nào cho đơn giản?** Thay mọi dấu `+`, `-`, `*` bằng dấu cách rồi `split()`:

```text
"1*05+3"  --thay toán tử bằng ' '-->  "1 05 3"  --split()-->  ["1", "05", "3"]
```

Thấy `"05"` không hợp lệ ⇒ loại biểu thức `1*05+3`.

Cách này **chậm** nhưng **gần như không thể sai** (Python lo phần tính toán), nên ta giữ nó làm **trọng tài** để so sánh các cách sau.

In [8]:
def split_operands(expr: str) -> List[str]:
    for op in "+-*":
        expr = expr.replace(op, " ")
    return expr.split()


def has_leading_zero(expr: str) -> bool:
    return any(not is_valid_operand(x) for x in split_operands(expr))


def add_operators_bruteforce(num: str, target: int) -> List[str]:
    result = []
    for ops in product(OPS, repeat=len(num) - 1):
        expr = build_expression(num, ops)
        if has_leading_zero(expr):
            continue                     # vi phạm luật số 0 đứng đầu
        if eval(expr) == target:
            result.append(expr)
    return result


print(split_operands("1*05+3"), has_leading_zero("1*05+3"))
for num, target, expected in examples:
    got = add_operators_bruteforce(num, target)
    print(f"{num!r:<13} {target:<5} -> {got}  đúng? {same_answer(got, expected)}")

['1', '05', '3'] True
'123'         6     -> ['1+2+3', '1*2*3']  đúng? True
'232'         8     -> ['2+3*2', '2*3+2']  đúng? True


'3456237490'  9191  -> []  đúng? True
'105'         5     -> ['10-5', '1*0+5']  đúng? True
'00'          0     -> ['0+0', '0-0', '0*0']  đúng? True


### Trace Cách 1 với `num = "105"`, `target = 5`

`"105"` có 2 khe ⇒ `4^2 = 16` bộ `ops`. Trong trace, ký hiệu `·` nghĩa là **nối** (lựa chọn `''`). Thử dự đoán trước bằng tay:

- Các biểu thức có khe 1 là `+`, `-`, `*` **và** khe 2 là "nối" sẽ chứa toán hạng `"05"` ⇒ **bị loại**: `1+05`, `1-05`, `1*05` (3 biểu thức).
- 13 biểu thức còn lại được `eval`. Chỉ có `10-5 = 5` và `1*0+5 = 0 + 5 = 5` là bằng target.

Lưu ý `1*05` có "giá trị" `1 * 5 = 5` — nếu quên lọc số 0 đứng đầu, ta sẽ trả thừa biểu thức này (xem mục Lỗi thường gặp).

In [9]:
def trace_bruteforce(num: str, target: int) -> List[str]:
    print(f"num = {num!r}, target = {target}, số bộ ops = 4^{len(num) - 1} = {4 ** (len(num) - 1)}")
    found = []
    for k, ops in enumerate(product(OPS, repeat=len(num) - 1), start=1):
        expr = build_expression(num, ops)
        shown = " ".join(op if op else "·" for op in ops)    # '·' = nối
        if has_leading_zero(expr):
            print(f"{k:>2}. khe: {shown}   {expr:<6} -> LOẠI (số 0 đứng đầu)")
            continue
        value = eval(expr)
        mark = "  ✔ KHỚP" if value == target else ""
        print(f"{k:>2}. khe: {shown}   {expr:<6} = {value}{mark}")
        if value == target:
            found.append(expr)
    print("Kết quả:", found)
    return found

trace_bruteforce("105", 5);

num = '105', target = 5, số bộ ops = 4^2 = 16
 1. khe: · ·   105    = 105
 2. khe: · +   10+5   = 15
 3. khe: · -   10-5   = 5  ✔ KHỚP
 4. khe: · *   10*5   = 50
 5. khe: + ·   1+05   -> LOẠI (số 0 đứng đầu)
 6. khe: + +   1+0+5  = 6
 7. khe: + -   1+0-5  = -4
 8. khe: + *   1+0*5  = 1
 9. khe: - ·   1-05   -> LOẠI (số 0 đứng đầu)
10. khe: - +   1-0+5  = 6
11. khe: - -   1-0-5  = -4
12. khe: - *   1-0*5  = 1
13. khe: * ·   1*05   -> LOẠI (số 0 đứng đầu)
14. khe: * +   1*0+5  = 5  ✔ KHỚP
15. khe: * -   1*0-5  = -5
16. khe: * *   1*0*5  = 0
Kết quả: ['10-5', '1*0+5']


### Độ phức tạp Cách 1 — và vì sao nó chậm

Gọi `n = len(num)`.

- Số biểu thức sinh ra: đúng `4^(n-1)`.
- Mỗi biểu thức tốn `O(n)` để dựng chuỗi, `O(n)` để tách toán hạng, và `O(n)` để `eval` (parse + tính).

⇒ **Thời gian `O(n · 4^(n-1))`**, **bộ nhớ phụ `O(n)`** cho một biểu thức đang xét (không tính danh sách kết quả).

Với `n = 10` là `262144` lần `eval` — chạy được nhưng mất cỡ vài giây trong Python, dễ bị Time Limit Exceeded. Ba chỗ lãng phí:

1. **Sinh cả biểu thức hỏng rồi mới loại.** Ví dụ `"105"`: mọi bộ `ops` có "khe 1 là toán tử, khe 2 là nối" đều tạo `"05"`. Với chuỗi dài và nhiều số 0, số biểu thức hỏng rất lớn — ta tốn công dựng rồi vứt.
2. **Làm lại từ đầu cho mỗi biểu thức.** `1+2+3` và `1+2-3` có chung tiền tố `1+2`, nhưng mỗi lần `eval` lại tính `1+2` từ đầu.
3. **`eval` đắt:** mỗi lần phải parse chuỗi.

Cell dưới đo thời gian: mỗi khi thêm **1 chữ số**, số biểu thức **gấp 4**, nên thời gian cũng gấp khoảng 4 (con số cụ thể tùy máy).

In [10]:
import time

for n in [6, 7, 8, 9]:
    s = "123456789"[:n]
    start = time.perf_counter()
    add_operators_bruteforce(s, 45)
    elapsed = time.perf_counter() - start
    print(f"n = {n}: {4 ** (n - 1):>6} biểu thức, brute force mất {elapsed:.3f} giây")

n = 6:   1024 biểu thức, brute force mất 0.008 giây
n = 7:   4096 biểu thức, brute force mất 0.034 giây
n = 8:  16384 biểu thức, brute force mất 0.143 giây


n = 9:  65536 biểu thức, brute force mất 0.688 giây


## 4. Kiến thức nền: đệ quy và backtracking (quay lui)

### 4.1. Đệ quy (recursion) là gì?

**Đệ quy** là khi một hàm **gọi lại chính nó** để giải một bài toán **nhỏ hơn** cùng loại. Mỗi hàm đệ quy cần:

- **Base case (trường hợp cơ sở):** khi bài toán đủ nhỏ thì trả lời ngay, không gọi tiếp — nếu thiếu, hàm gọi mãi không dừng.
- **Bước đệ quy:** làm một phần việc, rồi giao phần còn lại cho lời gọi đệ quy.

### 4.2. Backtracking là gì?

**Backtracking (quay lui)** là cách liệt kê mọi lời giải bằng cách đi trên một **cây quyết định**: ở mỗi bước ta **chọn** một khả năng, **đi tiếp** (đệ quy), rồi **gỡ lựa chọn đó ra** để thử khả năng kế tiếp. Nhịp ba bước:

```text
chọn  ->  đi tiếp (đệ quy)  ->  gỡ ra (quay lui)
```

### 4.3. Ví dụ mini: liệt kê mọi cách cắt chuỗi thành các toán hạng

Bài này về bản chất gồm 2 quyết định: **cắt `num` thành các toán hạng** và **chọn toán tử giữa chúng**. Ta luyện phần "cắt" trước.

Hàm `dfs(index)` nghĩa là: "phần `num[:index]` đã cắt xong và nằm trong `path`; hãy cắt tiếp phần `num[index:]`".

- **Base case:** `index == len(s)` ⇒ đã cắt hết, lưu một bản sao của `path`.
- **Bước đệ quy:** thử mọi toán hạng tiếp theo `s[index:end + 1]` với `end` chạy từ `index` tới cuối: `append` vào `path` → `dfs(end + 1)` → `pop` ra.

`"123"` có 2 khe, mỗi khe "cắt" hoặc "không cắt" ⇒ `2^2 = 4` cách cắt. Nếu thêm luật số 0 đứng đầu và dùng `break` (mục 2.2), cách cắt `["1", "05"]` của `"105"` sẽ không bao giờ được sinh ra.

DFS (Depth-First Search — tìm kiếm theo chiều sâu) là tên gọi chung cho kiểu đi "xuống sâu hết một nhánh rồi mới quay lại": hàm đệ quy kiểu này hay được đặt tên `dfs`.

In [11]:
def all_splits(s: str, skip_leading_zero: bool = False) -> List[List[str]]:
    result = []
    path = []

    def dfs(index: int) -> None:
        if index == len(s):                  # base case: đã cắt hết
            result.append(path.copy())       # copy! vì path còn bị sửa tiếp
            return
        for end in range(index, len(s)):
            piece = s[index:end + 1]
            if skip_leading_zero and not is_valid_operand(piece):
                break                        # "05", "050"... đều hỏng -> dừng hẳn
            path.append(piece)               # chọn
            dfs(end + 1)                     # đi tiếp
            path.pop()                       # gỡ ra (quay lui)

    dfs(0)
    return result

print(all_splits("123"))                          # 4 cách = 2^(3-1)
print(all_splits("105"))                          # có cả ['1', '05']
print(all_splits("105", skip_leading_zero=True))  # ['1', '05'] bị cắt bỏ

[['1', '2', '3'], ['1', '23'], ['12', '3'], ['123']]
[['1', '0', '5'], ['1', '05'], ['10', '5'], ['105']]
[['1', '0', '5'], ['10', '5'], ['105']]


## 5. Cách 2 — Backtracking xây chuỗi biểu thức, `eval` ở cuối

### Cách 1 vướng ở đâu?

Cách 1 sinh **cả biểu thức hỏng** rồi mới loại, và mỗi biểu thức được xây lại từ đầu.

### Mấu chốt cải tiến

Ghép **"cắt toán hạng"** (mục 4.3) với **"chọn toán tử"** thành một hàm đệ quy duy nhất `dfs(index, expr)`:

- `expr` là biểu thức đã xây cho phần `num[:index]`.
- Thử mọi toán hạng tiếp theo `cur_str = num[index:end + 1]`; gặp toán hạng có số 0 đứng đầu thì `break` ⇒ **biểu thức hỏng không bao giờ được sinh ra** (cắt tỉa — pruning).
- Nếu `index == 0` thì đây là **số đầu tiên**: chưa có gì bên trái nên **không đặt toán tử**, chỉ `dfs(end + 1, cur_str)`.
- Ngược lại, thử lần lượt 3 toán tử: `dfs(end + 1, expr + op + cur_str)` với `op` thuộc `"+-*"`.
- Base case `index == n`: biểu thức đã dùng hết chữ số ⇒ `eval` và so với `target`.

Vì `expr + op + cur_str` tạo ra **chuỗi mới** (chuỗi trong Python là bất biến — immutable), lời gọi con không làm hỏng `expr` của lời gọi cha, nên ở đây **không cần bước "gỡ ra"** như `path.pop()`: quay lui xảy ra "tự động".

Lưu ý lựa chọn "nối" không còn là một toán tử riêng: nó nằm sẵn trong việc chọn **độ dài** toán hạng (`end` chạy xa hơn).

In [12]:
def add_operators_backtrack_eval(num: str, target: int) -> List[str]:
    n = len(num)
    result = []

    def dfs(index: int, expr: str) -> None:
        if index == n:                       # dùng hết chữ số -> chấm điểm bằng eval
            if eval(expr) == target:
                result.append(expr)
            return
        for end in range(index, n):
            cur_str = num[index:end + 1]
            if not is_valid_operand(cur_str):
                break                        # cắt tỉa số 0 đứng đầu
            if index == 0:
                dfs(end + 1, cur_str)        # số đầu tiên: không có toán tử
            else:
                for op in "+-*":
                    dfs(end + 1, expr + op + cur_str)

    dfs(0, "")
    return result


for num, target, expected in examples:
    got = add_operators_backtrack_eval(num, target)
    print(f"{num!r:<13} {target:<5} -> {got}  đúng? {same_answer(got, expected)}")

# Số biểu thức hoàn chỉnh (lá) được eval: Cách 1 vs Cách 2
def count_leaves_cach2(num: str) -> int:
    count = 0
    def dfs(index: int) -> None:
        nonlocal count
        if index == len(num):
            count += 1
            return
        for end in range(index, len(num)):
            if not is_valid_operand(num[index:end + 1]):
                break
            for _ in range(1 if index == 0 else 3):
                dfs(end + 1)
    dfs(0)
    return count

for s in ["105", "1000", "1000000009"]:
    print(f"{s!r:<13} Cách 1 xét {4 ** (len(s) - 1):>6} biểu thức, Cách 2 chỉ eval {count_leaves_cach2(s):>6}")

'123'         6     -> ['1+2+3', '1*2*3']  đúng? True
'232'         8     -> ['2+3*2', '2*3+2']  đúng? True


'3456237490'  9191  -> []  đúng? True
'105'         5     -> ['1*0+5', '10-5']  đúng? True
'00'          0     -> ['0+0', '0-0', '0*0']  đúng? True
'105'         Cách 1 xét     16 biểu thức, Cách 2 chỉ eval     13
'1000'        Cách 1 xét     64 biểu thức, Cách 2 chỉ eval     40
'1000000009'  Cách 1 xét 262144 biểu thức, Cách 2 chỉ eval  29524


### Cách 2 còn vướng gì? Thử tính giá trị **dần dần**

Cách 2 bỏ được biểu thức hỏng, nhưng với chuỗi **không có số 0** (như `"9999999999"`) thì số lá vẫn là `4^(n-1)` và **mỗi lá vẫn phải `eval`** cả chuỗi từ đầu.

Ý tưởng tự nhiên: mang theo **giá trị hiện tại** `value` của `expr` trong lời gọi đệ quy, rồi cập nhật ngay khi thêm toán hạng:

```text
'+' cur:  value = value + cur
'-' cur:  value = value - cur
'*' cur:  value = value * cur      <- ???
```

Với `+` và `-` thì đúng. Nhưng phép `*` thì **sai**, vì nó bỏ qua độ ưu tiên: với `1+2*3`, sau `1+2` ta có `value = 3`, rồi `*3` cho `3 * 3 = 9` — tức là đang tính `(1+2)*3`. Đáp án đúng là `7`.

Cell dưới viết bản "ngây thơ" này (**sai có chủ đích**) để thấy nó trả thừa `"1+2*3"` cho `target = 9`. Đáp án đúng của `"123", 9` chỉ là `["12-3"]`.

In [13]:
def add_operators_naive_value(num: str, target: int) -> List[str]:
    # SAI CÓ CHỦ ĐÍCH: phép nhân không tôn trọng độ ưu tiên
    n = len(num)
    result = []

    def dfs(index: int, expr: str, value: int) -> None:
        if index == n:
            if value == target:
                result.append(expr)
            return
        for end in range(index, n):
            cur_str = num[index:end + 1]
            if not is_valid_operand(cur_str):
                break
            cur = int(cur_str)
            if index == 0:
                dfs(end + 1, cur_str, cur)
            else:
                dfs(end + 1, expr + "+" + cur_str, value + cur)
                dfs(end + 1, expr + "-" + cur_str, value - cur)
                dfs(end + 1, expr + "*" + cur_str, value * cur)   # <- lỗi ở đây

    dfs(0, "", 0)
    return result

print("ngây thơ :", add_operators_naive_value("123", 9))
print("trọng tài:", add_operators_bruteforce("123", 9))
print("eval('1+2*3') =", eval("1+2*3"), "nhưng bản ngây thơ tính ra (1+2)*3 =", (1 + 2) * 3)

ngây thơ : ['1+2*3', '12-3']
trọng tài: ['12-3']
eval('1+2*3') = 7 nhưng bản ngây thơ tính ra (1+2)*3 = 9


## 6. Mấu chốt của bài: nhớ thêm **hạng tử cuối** `prev`

Nhớ lại mục 2.3: giá trị biểu thức = **tổng các hạng tử có dấu**, và phép `*` **chỉ dính vào hạng tử cuối**. Vậy ngoài `value`, ta giữ thêm:

- `value`: giá trị của **cả** biểu thức hiện tại (đã tính đúng ưu tiên).
- `prev`: giá trị của **hạng tử cuối cùng**, **kèm dấu** của nó (last operand / last term).

Khi đó: `value = (tổng các hạng tử phía trước) + prev`.

Quy tắc cập nhật khi thêm toán hạng `cur`:

| Thêm | `value` mới | `prev` mới | Vì sao |
|---|---|---|---|
| `+cur` | `value + cur` | `cur` | mở một hạng tử mới là `+cur` |
| `-cur` | `value - cur` | `-cur` | mở một hạng tử mới là `-cur` (**nhớ dấu âm**) |
| `*cur` | `value - prev + prev * cur` | `prev * cur` | hạng tử cuối `prev` "lớn lên" thành `prev * cur` |
| số đầu tiên `cur` | `cur` | `cur` | biểu thức chỉ có một hạng tử |

Đọc công thức phép nhân như một thao tác **"hoàn tác rồi làm lại"**:

1. `value - prev`: **gỡ** hạng tử cuối ra khỏi tổng (còn lại tổng các hạng tử phía trước).
2. `prev * cur`: hạng tử cuối sau khi nhân thêm `cur`.
3. Cộng lại: `value - prev + prev * cur`.

**Ví dụ số `1+2*3*4`** (đáp án `eval` là `25`):

| Bước | Thêm | Tính `value` | `value` | `prev` | Các hạng tử lúc này |
|---:|---|---|---:|---:|---|
| 1 | `1` (số đầu) | `1` | 1 | 1 | `+1` |
| 2 | `+2` | `1 + 2` | 3 | 2 | `+1`, `+2` |
| 3 | `*3` | `3 - 2 + 2*3` | 7 | 6 | `+1`, `+(2*3)` |
| 4 | `*4` | `7 - 6 + 6*4` | 25 | 24 | `+1`, `+(2*3*4)` |

Ở bước 3, `3 - 2 = 1` là phần "phía trước" (`+1`), còn hạng tử cuối `2` lớn lên thành `6`. Ở bước 4, `7 - 6 = 1` vẫn là phần phía trước, hạng tử cuối `6` lớn lên thành `24`. Kết quả `25` ✔ khớp `eval`.

In [14]:
def apply_op(value: int, prev: int, op: str, cur: int):
    # Trả về (value mới, prev mới) khi nối thêm "op cur" vào biểu thức
    if op == "+":
        return value + cur, cur
    if op == "-":
        return value - cur, -cur
    if op == "*":
        return value - prev + prev * cur, prev * cur
    raise ValueError(op)


def tokenize(expr: str):
    # "1+2*3" -> [('', 1), ('+', 2), ('*', 3)]   ('' = số đầu tiên)
    tokens, op, digits = [], "", ""
    for ch in expr:
        if ch in "+-*":
            tokens.append((op, int(digits)))
            op, digits = ch, ""
        else:
            digits += ch
    tokens.append((op, int(digits)))
    return tokens


def simulate(expr: str, verbose: bool = True) -> int:
    tokens = tokenize(expr)
    value = prev = tokens[0][1]                 # số đầu tiên
    if verbose:
        print(f"số đầu {value:<3} -> value = {value:<4} prev = {prev}")
    for op, cur in tokens[1:]:
        old_value, old_prev = value, prev
        value, prev = apply_op(value, prev, op, cur)
        if verbose:
            how = f"{old_value} - ({old_prev}) + ({old_prev})*{cur}" if op == "*" else f"{old_value} {op} {cur}"
            print(f"thêm {op}{cur:<4} -> value = {value:<4} prev = {prev:<4} (tính: {how})")
    return value

result = simulate("1+2*3*4")
print("simulate =", result, "| eval =", eval("1+2*3*4"))

số đầu 1   -> value = 1    prev = 1
thêm +2    -> value = 3    prev = 2    (tính: 1 + 2)
thêm *3    -> value = 7    prev = 6    (tính: 3 - (2) + (2)*3)
thêm *4    -> value = 25   prev = 24   (tính: 7 - (6) + (6)*4)
simulate = 25 | eval = 25


### Vì sao `prev` phải mang **dấu**? Ví dụ `5-2*3`

`eval("5-2*3") = 5 - 6 = -1`. Hạng tử cuối của `5-2` là **`-2`** chứ không phải `2`:

| Bước | Thêm | Tính `value` | `value` | `prev` |
|---:|---|---|---:|---:|
| 1 | `5` (số đầu) | `5` | 5 | 5 |
| 2 | `-2` | `5 - 2` | 3 | **-2** |
| 3 | `*3` | `3 - (-2) + (-2)*3 = 3 + 2 - 6` | -1 | -6 |

Nếu lỡ lưu `prev = 2` (quên dấu), bước 3 thành `3 - 2 + 2*3 = 7` — sai. Lý do: `value - prev` phải trả về đúng "phần phía trước" là `5`, mà `3 - (-2) = 5` chỉ đúng khi `prev = -2`.

### Vì sao công thức luôn đúng? (bất biến — invariant)

Ta luôn giữ được điều sau, gọi là **bất biến** (invariant — tính chất đúng ở mọi bước):

> `value` = (tổng các hạng tử **trước** hạng tử cuối) + `prev`

- Sau số đầu tiên: không có hạng tử nào phía trước, `value = prev = cur` ✔.
- Thêm `+cur` hoặc `-cur`: hạng tử cũ `prev` chuyển vào "phía trước", hạng tử mới là `±cur` ⇒ `value ± cur` và `prev = ±cur` ✔.
- Thêm `*cur`: "phía trước" **không đổi** và bằng `value - prev`; hạng tử cuối thành `prev * cur` ⇒ `value_mới = (value - prev) + prev * cur` ✔.

Cell dưới chạy thêm vài ví dụ (có cả chuỗi nhân ngay từ đầu `2*3*4`), rồi so `simulate` với `eval` trên 1000 biểu thức ngẫu nhiên.

In [15]:
import random

simulate("5-2*3")
print("eval('5-2*3') =", eval("5-2*3"))
print()
for expr in ["2*3*4", "1-2*3+4*5", "12*3-4*5+6", "7"]:
    print(f"{expr:<11} simulate = {simulate(expr, verbose=False):<5} eval = {eval(expr)}")

random.seed(282)
ok = 0
for _ in range(1000):
    length = random.randint(1, 8)
    digits = "".join(random.choice("0123456789") for _ in range(length))
    ops = [random.choice(OPS) for _ in range(length - 1)]
    expr = build_expression(digits, ops)
    if has_leading_zero(expr):
        expr = build_expression(digits.replace("0", "7"), ops)   # tránh số 0 đứng đầu cho eval
    ok += simulate(expr, verbose=False) == eval(expr)
print(f"simulate khớp eval: {ok}/1000")

số đầu 5   -> value = 5    prev = 5
thêm -2    -> value = 3    prev = -2   (tính: 5 - 2)
thêm *3    -> value = -1   prev = -6   (tính: 3 - (-2) + (-2)*3)
eval('5-2*3') = -1

2*3*4       simulate = 24    eval = 24
1-2*3+4*5   simulate = 15    eval = 15
12*3-4*5+6  simulate = 22    eval = 22
7           simulate = 7     eval = 7
simulate khớp eval: 1000/1000


## 7. Cách 3 — Backtracking giữ `value` và `prev` (cách chính)

Giờ ghép tất cả lại. Hàm đệ quy:

```python
dfs(index, path, value, prev)
```

| Tham số | Ý nghĩa |
|---|---|
| `index` | đã dùng xong `num[:index]`; toán hạng tiếp theo bắt đầu tại `index` |
| `path` | chuỗi biểu thức đã xây cho `num[:index]` (để trả về khi khớp) |
| `value` | giá trị đúng ưu tiên của `path` |
| `prev` | giá trị hạng tử cuối của `path`, kèm dấu |

Các bước trong thân hàm:

1. **Base case** `index == n`: dùng hết chữ số. Nếu `value == target` thì lưu `path`. Không cần `eval` nữa!
2. **Vòng `end` từ `index` đến `n - 1`**: toán hạng tiếp theo là `cur_str = num[index:end + 1]`.
   - Nếu `cur_str` có số 0 đứng đầu ⇒ **`break`** (mọi toán hạng dài hơn cũng hỏng, xem mục 2.2).
   - `cur = int(cur_str)` — an toàn vì đã kiểm luật số 0 trên chuỗi.
3. Nếu `index == 0` (số đầu tiên): `dfs(end + 1, cur_str, cur, cur)` — không có toán tử phía trước.
4. Ngược lại, rẽ 3 nhánh theo bảng ở mục 6:
   - `dfs(end + 1, path + "+" + cur_str, value + cur, cur)`
   - `dfs(end + 1, path + "-" + cur_str, value - cur, -cur)`
   - `dfs(end + 1, path + "*" + cur_str, value - prev + prev * cur, prev * cur)`

Lời gọi khởi đầu: `dfs(0, "", 0, 0)` — `value` và `prev` ban đầu không quan trọng vì nhánh `index == 0` ghi đè ngay.

Để ý "nối chữ số" được xử lý bởi vòng `end` (chọn toán hạng dài bao nhiêu), còn 3 toán tử được xử lý bởi 3 lời gọi đệ quy. Đúng 4 lựa chọn mỗi khe như ở Cách 1, chỉ là được tổ chức thành cây.

In [16]:
def add_operators_backtrack(num: str, target: int) -> List[str]:
    n = len(num)
    result = []

    def dfs(index: int, path: str, value: int, prev: int) -> None:
        if index == n:
            if value == target:
                result.append(path)
            return
        for end in range(index, n):
            cur_str = num[index:end + 1]
            if len(cur_str) > 1 and cur_str[0] == "0":
                break                                    # "05", "012"... -> dừng hẳn
            cur = int(cur_str)
            if index == 0:
                dfs(end + 1, cur_str, cur, cur)          # số đầu tiên
            else:
                dfs(end + 1, path + "+" + cur_str, value + cur, cur)
                dfs(end + 1, path + "-" + cur_str, value - cur, -cur)
                dfs(end + 1, path + "*" + cur_str, value - prev + prev * cur, prev * cur)

    dfs(0, "", 0, 0)
    return result


for num, target, expected in examples:
    got = add_operators_backtrack(num, target)
    print(f"{num!r:<13} {target:<5} -> {got}  đúng? {same_answer(got, expected)}")

'123'         6     -> ['1+2+3', '1*2*3']  đúng? True
'232'         8     -> ['2+3*2', '2*3+2']  đúng? True
'3456237490'  9191  -> []  đúng? True
'105'         5     -> ['1*0+5', '10-5']  đúng? True
'00'          0     -> ['0+0', '0-0', '0*0']  đúng? True


### Trace cây gọi cho `num = "123"`, `target = 6`

Hàm `trace_backtrack` dưới đây giống hệt Cách 3 nhưng **in mỗi lời gọi** `dfs` trên một dòng:

- **Thụt lề** = độ sâu trong cây gọi (mỗi mức sâu hơn thụt thêm 2 dấu cách).
- Nhãn đầu dòng cho biết lời gọi này được tạo từ đâu: `số đầu '12'` (toán hạng đầu tiên), hoặc `'+3'`, `'-3'`, `'*3'` (toán tử + toán hạng vừa thêm).
- Sau đó là trạng thái **sau khi** thêm: `path`, `value`, `prev`.
- Với nhánh `*`, cuối dòng in thêm phép tính `value - prev + prev*cur` để thấy công thức hoạt động.
- Dòng là **lá** (đã dùng hết chữ số) được đánh dấu `✔ KHỚP` hoặc `✘`.

Đọc theo thứ tự từ trên xuống chính là thứ tự DFS: đi sâu hết nhánh `'1'` → `'+2'` → `'+3'`, rồi mới quay lui thử `'-3'`, `'*3'`, …

In [17]:
def trace_backtrack(num: str, target: int) -> List[str]:
    n = len(num)
    result = []

    def fmt(x: int) -> str:
        return f"({x})" if x < 0 else str(x)

    def show(depth: int, label: str, index: int, path: str, value: int, prev: int, extra: str = "") -> None:
        left = "  " * depth + label
        mark = ""
        if index == n:
            mark = "  ✔ KHỚP" if value == target else "  ✘"
        print(f"{left:<16} path={path!r:<9} value={value:<4} prev={prev:<4}{extra}{mark}")

    def dfs(index: int, path: str, value: int, prev: int, depth: int) -> None:
        if index == n:
            if value == target:
                result.append(path)
            return
        for end in range(index, n):
            cur_str = num[index:end + 1]
            if len(cur_str) > 1 and cur_str[0] == "0":
                print("  " * (depth + 1) + f"✂ {cur_str!r}: số 0 đứng đầu -> break")
                break
            cur = int(cur_str)
            if index == 0:
                show(depth + 1, f"số đầu {cur_str!r}", end + 1, cur_str, cur, cur)
                dfs(end + 1, cur_str, cur, cur, depth + 1)
            else:
                for op in "+-*":
                    if op == "+":
                        nv, np_, extra = value + cur, cur, ""
                    elif op == "-":
                        nv, np_, extra = value - cur, -cur, ""
                    else:
                        nv, np_ = value - prev + prev * cur, prev * cur
                        extra = f" ({value}-{fmt(prev)}+{fmt(prev)}*{cur})"
                    show(depth + 1, f"'{op}{cur_str}'", end + 1, path + op + cur_str, nv, np_, extra)
                    dfs(end + 1, path + op + cur_str, nv, np_, depth + 1)

    print(f"dfs(0, '', 0, 0)   num = {num!r}, target = {target}")
    dfs(0, "", 0, 0, 0)
    print("Kết quả:", result)
    return result

trace_backtrack("123", 6);

dfs(0, '', 0, 0)   num = '123', target = 6
  số đầu '1'     path='1'       value=1    prev=1   
    '+2'         path='1+2'     value=3    prev=2   
      '+3'       path='1+2+3'   value=6    prev=3     ✔ KHỚP
      '-3'       path='1+2-3'   value=0    prev=-3    ✘
      '*3'       path='1+2*3'   value=7    prev=6    (3-2+2*3)  ✘
    '-2'         path='1-2'     value=-1   prev=-2  
      '+3'       path='1-2+3'   value=2    prev=3     ✘
      '-3'       path='1-2-3'   value=-4   prev=-3    ✘
      '*3'       path='1-2*3'   value=-5   prev=-6   (-1-(-2)+(-2)*3)  ✘
    '*2'         path='1*2'     value=2    prev=2    (1-1+1*2)
      '+3'       path='1*2+3'   value=5    prev=3     ✘
      '-3'       path='1*2-3'   value=-1   prev=-3    ✘
      '*3'       path='1*2*3'   value=6    prev=6    (2-2+2*3)  ✔ KHỚP
    '+23'        path='1+23'    value=24   prev=23    ✘
    '-23'        path='1-23'    value=-22  prev=-23   ✘
    '*23'        path='1*23'    value=23   prev=23   (1-1+1*23)  ✘
  số 

### Đọc trace `"123"` bằng bảng

Nhánh dẫn tới hai đáp án:

| Đường đi trong cây | `path` | `value` | `prev` | Giải thích |
|---|---|---:|---:|---|
| `số đầu '1'` | `1` | 1 | 1 | số đầu: `value = prev = 1` |
| → `'+2'` | `1+2` | 3 | 2 | mở hạng tử mới `+2` |
| → → `'+3'` | `1+2+3` | 6 | 3 | lá, `6 == 6` ✔ |
| → → `'*3'` | `1+2*3` | 7 | 6 | `3 - 2 + 2*3 = 7`, lá ✘ — **đúng ưu tiên**, không phải 9 |
| → `'*2'` | `1*2` | 2 | 2 | `1 - 1 + 1*2 = 2` (nhân ngay số đầu cũng đúng) |
| → → `'*3'` | `1*2*3` | 6 | 6 | `2 - 2 + 2*3 = 6`, lá ✔ |
| → `'-2'` → `'*3'` | `1-2*3` | -5 | -6 | `-1 - (-2) + (-2)*3 = -5` = `eval("1-2*3")` |

Một số quan sát:

- Có **16 lá** = `4^2` — đúng bằng số biểu thức của Cách 1, vì `"123"` không có số 0 nên không nhánh nào bị cắt.
- Nhánh `số đầu '12'` và `số đầu '123'` là các lựa chọn "nối" ở khe đầu tiên. Lá `path='123'` là biểu thức không có toán tử nào.
- Kết quả thu được theo thứ tự DFS là `['1+2+3', '1*2*3']` — khác thứ tự trong đề `["1*2*3","1+2+3"]`, nhưng đề chấp nhận thứ tự bất kỳ.

### Trace thứ hai: `num = "105"`, `target = 5` — nhìn thấy nhánh bị cắt

Ví dụ này có số `0` ở giữa. Hãy để ý dòng `✂`: khi đứng ở `index = 1` (chữ số `'0'`), toán hạng `"0"` được thử, nhưng toán hạng `"05"` bị chặn ngay và vòng `end` **dừng hẳn**. Cũng hãy để ý `prev = 0` sau `'+0'`, `'-0'`, `'*0'` — rồi xem phép nhân với `prev = 0` cho kết quả thế nào.

In [18]:
trace_backtrack("105", 5);

dfs(0, '', 0, 0)   num = '105', target = 5
  số đầu '1'     path='1'       value=1    prev=1   
    '+0'         path='1+0'     value=1    prev=0   
      '+5'       path='1+0+5'   value=6    prev=5     ✘
      '-5'       path='1+0-5'   value=-4   prev=-5    ✘
      '*5'       path='1+0*5'   value=1    prev=0    (1-0+0*5)  ✘
    '-0'         path='1-0'     value=1    prev=0   
      '+5'       path='1-0+5'   value=6    prev=5     ✘
      '-5'       path='1-0-5'   value=-4   prev=-5    ✘
      '*5'       path='1-0*5'   value=1    prev=0    (1-0+0*5)  ✘
    '*0'         path='1*0'     value=0    prev=0    (1-1+1*0)
      '+5'       path='1*0+5'   value=5    prev=5     ✔ KHỚP
      '-5'       path='1*0-5'   value=-5   prev=-5    ✘
      '*5'       path='1*0*5'   value=0    prev=0    (0-0+0*5)  ✘
    ✂ '05': số 0 đứng đầu -> break
  số đầu '10'    path='10'      value=10   prev=10  
    '+5'         path='10+5'    value=15   prev=5     ✘
    '-5'         path='10-5'    value=5    prev=-5  

### Đọc trace `"105"` bằng bảng

| Lá | Tính dần | `value` | Khớp 5? |
|---|---|---:|:---:|
| `1+0+5` | `1` → `1+0=1` (prev 0) → `1+5=6` | 6 | |
| `1+0*5` | … → `1 - 0 + 0*5 = 1` | 1 | |
| `1*0+5` | `1-1+1*0 = 0` (prev 0) → `0 + 5` | 5 | ✔ |
| `1*0*5` | `0 - 0 + 0*5 = 0` | 0 | |
| `10-5` | số đầu `10` → `10 - 5` | 5 | ✔ |
| `10*5` | `10 - 10 + 10*5 = 50` | 50 | |
| `105` | số đầu `105`, không toán tử | 105 | |

Những điều đáng chú ý:

1. **Nhánh `"05"` bị cắt ngay tại gốc của nó**: dòng `✂ '05': số 0 đứng đầu -> break` xuất hiện **một lần**, và nhờ vậy 3 biểu thức `1+05`, `1-05`, `1*05` **không bao giờ được tạo ra**. Cách 3 chỉ có **13 lá**, ít hơn 16 bộ `ops` của Cách 1 (Cách 1 phải dựng 3 biểu thức hỏng rồi mới loại).
2. **Toán hạng `"0"` đơn lẻ vẫn được dùng** (`1*0+5` là đáp án!). Điều kiện `len(cur_str) > 1` là bắt buộc — nếu chỉ kiểm `cur_str[0] == "0"` ta sẽ mất đáp án này.
3. Sau `'-0'`, `prev = -0 = 0` — trong số nguyên, `-0` chính là `0`, nên không có gì đặc biệt.
4. Kết quả `['1*0+5', '10-5']` khớp với ví dụ.

## 8. Cách 4 (tùy chọn) — `path` dạng list với `append` / `pop`

### Cách 3 còn tốn ở đâu?

Mỗi lời gọi tạo chuỗi mới `path + op + cur_str`. Vì chuỗi bất biến, Python phải **sao chép** toàn bộ `path` cũ — tốn `O(n)` mỗi lần, và việc này xảy ra ở **mọi nút** của cây, kể cả hàng trăm nghìn nút không bao giờ thành đáp án. Ngoài ra `num[index:end + 1]` rồi `int(...)` cũng làm lại việc đổi chuỗi thành số cho mỗi `end`.

### Mấu chốt cải tiến

- Giữ `path` là **một list dùng chung** các mảnh (`'1'`, `'+'`, `'23'`, …). Trước khi đi xuống: `append` toán tử và toán hạng; sau khi quay về: `pop` hai lần. Đây đúng là nhịp **chọn → đi tiếp → gỡ ra** ở mục 4.2. Chỉ khi **khớp target** ở lá mới `"".join(path)` để tạo chuỗi.
- Tính `cur` **dần dần** bằng `cur = cur * 10 + int(num[end])` (mục 2.1) thay vì `int(num[index:end + 1])` mỗi vòng.
- Kiểm số 0 đứng đầu bằng `end > index and num[index] == "0"`: toán hạng dài hơn 1 chữ số (`end > index`) mà bắt đầu bằng `'0'`.

Vì phải `pop` đúng số lần đã `append`, cách này dễ sai hơn một chút — đó là lý do notebook dùng Cách 3 làm cách chính. Báo trước: với `n <= 10` trong Python, lợi ích **đo được** của Cách 4 rất nhỏ (xem mục 9); giá trị chính của nó là tư duy "dùng chung một `path`, chọn → đi tiếp → gỡ ra" và bộ nhớ `O(n)` thay vì `O(n^2)`.

In [19]:
def add_operators_fast(num: str, target: int) -> List[str]:
    n = len(num)
    result = []
    path = []                                   # list dùng chung, sửa tại chỗ

    def dfs(index: int, value: int, prev: int) -> None:
        if index == n:
            if value == target:
                result.append("".join(path))    # chỉ tạo chuỗi khi khớp
            return
        cur = 0
        for end in range(index, n):
            if end > index and num[index] == "0":
                break                           # số 0 đứng đầu
            cur = cur * 10 + int(num[end])      # ghép chữ số dần dần
            cur_str = num[index:end + 1]
            if index == 0:
                path.append(cur_str)
                dfs(end + 1, cur, cur)
                path.pop()
            else:
                for op, new_value, new_prev in (
                    ("+", value + cur, cur),
                    ("-", value - cur, -cur),
                    ("*", value - prev + prev * cur, prev * cur),
                ):
                    path.append(op)             # chọn
                    path.append(cur_str)
                    dfs(end + 1, new_value, new_prev)   # đi tiếp
                    path.pop()                  # gỡ ra
                    path.pop()

    dfs(0, 0, 0)
    return result


for num, target, expected in examples:
    got = add_operators_fast(num, target)
    print(f"{num!r:<13} {target:<5} -> {got}  đúng? {same_answer(got, expected)}")

'123'         6     -> ['1+2+3', '1*2*3']  đúng? True
'232'         8     -> ['2+3*2', '2*3+2']  đúng? True


'3456237490'  9191  -> []  đúng? True
'105'         5     -> ['1*0+5', '10-5']  đúng? True
'00'          0     -> ['0+0', '0-0', '0*0']  đúng? True


### Trace `path` list với `num = "12"`, `target = 2`

Ví dụ nhỏ để thấy `path` **phồng lên rồi xẹp xuống** đúng nhịp. Dự đoán: có 4 lá là `1+2 = 3`, `1-2 = -1`, `1*2 = 2` ✔ và `12`. Sau mỗi lá, hai lần `pop` trả `path` về `['1']`; sau khi xong mọi nhánh của `'1'`, một lần `pop` trả về `[]` rồi mới thử số đầu `'12'`.

Điểm quan trọng: **sau khi duyệt xong, `path` phải trở về rỗng `[]`** — nếu không, ta đã `append`/`pop` lệch nhau.

In [20]:
def trace_path_list(num: str, target: int) -> List[str]:
    n = len(num)
    result = []
    path = []

    def dfs(index: int, value: int, prev: int, depth: int) -> None:
        pad = "  " * depth
        if index == n:
            mark = "✔ KHỚP -> lưu " + repr("".join(path)) if value == target else "✘"
            print(f"{pad}lá: value={value} {mark}")
            if value == target:
                result.append("".join(path))
            return
        cur = 0
        for end in range(index, n):
            if end > index and num[index] == "0":
                break
            cur = cur * 10 + int(num[end])
            cur_str = num[index:end + 1]
            choices = [("", cur, cur)] if index == 0 else [
                ("+", value + cur, cur), ("-", value - cur, -cur), ("*", value - prev + prev * cur, prev * cur)]
            for op, nv, np_ in choices:
                pieces = [op, cur_str] if op else [cur_str]
                path.extend(pieces)
                print(f"{pad}append {pieces} -> path = {path}")
                dfs(end + 1, nv, np_, depth + 1)
                for _ in pieces:
                    path.pop()
                print(f"{pad}pop    -> path = {path}")

    dfs(0, 0, 0, 0)
    print("Kết quả:", result, "| path cuối cùng:", path)
    return result

trace_path_list("12", 2);

append ['1'] -> path = ['1']
  append ['+', '2'] -> path = ['1', '+', '2']
    lá: value=3 ✘
  pop    -> path = ['1']
  append ['-', '2'] -> path = ['1', '-', '2']
    lá: value=-1 ✘
  pop    -> path = ['1']
  append ['*', '2'] -> path = ['1', '*', '2']
    lá: value=2 ✔ KHỚP -> lưu '1*2'
  pop    -> path = ['1']
pop    -> path = []
append ['12'] -> path = ['12']
  lá: value=12 ✘
pop    -> path = []
Kết quả: ['1*2'] | path cuối cùng: []


## 9. So sánh tốc độ 4 cách trên cùng input

Ta chạy cả 4 cách trên vài chuỗi 9–10 chữ số:

- `"123456789"`, target `45`: chuỗi không có số 0, có rất nhiều đáp án.
- `"9999999999"`, target `0`: chuỗi 10 chữ số **không có số 0** ⇒ `4^9 = 262144` lá — trường hợp nặng nhất (Cách 1 được **bỏ qua** ở chuỗi này để notebook không chạy quá lâu).
- `"1000000009"`, target `0`: nhiều số 0 ⇒ cắt tỉa số 0 đứng đầu loại được rất nhiều nhánh.

Điều cần quan sát (con số cụ thể tùy máy):

1. Cách 1 và Cách 2 chậm vì `eval` ở mỗi lá. Cách 2 nhanh hơn Cách 1 **vài lần** khi chuỗi có nhiều số 0 (nhờ cắt tỉa: với `"1000000009"` chỉ còn 29524 lá thay vì 262144 bộ `ops`, như đã đếm ở mục 5), nhưng chỉ nhanh hơn **một ít** khi không có số 0.
2. Cách 3 nhanh hơn Cách 2 **hàng chục lần** — chỉ nhờ bỏ `eval` và tính `value` dần dần bằng `prev`. Đây là bước cải tiến quan trọng nhất của cả bài.
3. Cách 4 **ngang ngửa** Cách 3 trong Python (tùy máy, có lúc nhanh hơn chút, có lúc chậm hơn chút): chuỗi ở đây rất ngắn (tối đa 19 ký tự) nên sao chép chuỗi rẻ, còn mỗi lần `append`/`pop` cũng là một lời gọi hàm có chi phí. Ưu thế của `append/pop` chỉ rõ khi chuỗi dài hoặc ở các ngôn ngữ khác — một bài học thực tế: **tối ưu trên giấy chưa chắc nhanh hơn khi đo**, nên luôn đo trước khi kết luận.

In [21]:
def timed(fn, num, target):
    start = time.perf_counter()
    out = fn(num, target)
    return len(out), time.perf_counter() - start

methods = [
    ("Cách 1 brute+eval", add_operators_bruteforce),
    ("Cách 2 backtrack+eval", add_operators_backtrack_eval),
    ("Cách 3 value/prev", add_operators_backtrack),
    ("Cách 4 path list", add_operators_fast),
]

for num, target in [("123456789", 45), ("9999999999", 0), ("1000000009", 0)]:
    print(f"num = {num!r}, target = {target}")
    for name, fn in methods:
        if name.startswith("Cách 1") and len(num) == 10 and "0" not in num:
            print(f"   {name:<22} (bỏ qua: 262144 lần eval, chạy rất lâu)")
            continue
        count, sec = timed(fn, num, target)
        print(f"   {name:<22} {count:>5} đáp án   {sec:.3f} giây")

num = '123456789', target = 45


   Cách 1 brute+eval        121 đáp án   0.640 giây


   Cách 2 backtrack+eval    121 đáp án   0.509 giây
   Cách 3 value/prev        121 đáp án   0.017 giây
   Cách 4 path list         121 đáp án   0.022 giây
num = '9999999999', target = 0
   Cách 1 brute+eval      (bỏ qua: 262144 lần eval, chạy rất lâu)


   Cách 2 backtrack+eval   3930 đáp án   2.091 giây
   Cách 3 value/prev       3930 đáp án   0.068 giây
   Cách 4 path list        3930 đáp án   0.079 giây
num = '1000000009', target = 0


   Cách 1 brute+eval       3280 đáp án   0.783 giây


   Cách 2 backtrack+eval   3280 đáp án   0.258 giây
   Cách 3 value/prev       3280 đáp án   0.008 giây
   Cách 4 path list        3280 đáp án   0.009 giây


## 10. Code nộp LeetCode — Cách 3 (backtracking giữ `value` và `prev`)

Chuyển `add_operators_backtrack` vào đúng khuôn của LeetCode:

- Tên class `Solution`, phương thức `addOperators(self, num: str, target: int) -> List[str]`.
- `from typing import List` để Python hiểu chú thích kiểu `List[str]` (trên LeetCode dòng này đã có sẵn, tự chạy ở máy thì cần import).
- Hàm `dfs` lồng bên trong để dùng chung `num`, `target`, `n`, `result` mà không phải truyền qua tham số.
- Kiểm số 0 đứng đầu bằng `end > index and num[index] == "0"` — tương đương `len(cur_str) > 1 and cur_str[0] == "0"`, nhưng kiểm **trước khi** cắt chuỗi.

Nếu muốn nộp Cách 4, chỉ cần đặt thân hàm `add_operators_fast` vào trong `addOperators`.

In [22]:
from typing import List


class Solution:
    def addOperators(self, num: str, target: int) -> List[str]:
        n = len(num)
        result = []

        def dfs(index: int, path: str, value: int, prev: int) -> None:
            if index == n:
                if value == target:
                    result.append(path)
                return
            for end in range(index, n):
                if end > index and num[index] == "0":
                    break
                cur_str = num[index:end + 1]
                cur = int(cur_str)
                if index == 0:
                    dfs(end + 1, cur_str, cur, cur)
                else:
                    dfs(end + 1, path + "+" + cur_str, value + cur, cur)
                    dfs(end + 1, path + "-" + cur_str, value - cur, -cur)
                    dfs(end + 1, path + "*" + cur_str, value - prev + prev * cur, prev * cur)

        dfs(0, "", 0, 0)
        return result


print(Solution().addOperators("123", 6))
print(Solution().addOperators("232", 8))
print(Solution().addOperators("3456237490", 9191))

['1+2+3', '1*2*3']
['2+3*2', '2*3+2']
[]


### Bộ test bằng `assert` và test đối chiếu ngẫu nhiên

**Phần 1 — test cố định.** Ta kiểm tất cả 4 cách và `Solution` trên các nhóm:

| Nhóm | Input | Mong đợi | Kiểm điều gì |
|---|---|---|---|
| Ví dụ đề | `"123", 6` / `"232", 8` / `"3456237490", 9191` | như đề | đúng yêu cầu cơ bản |
| Số 0 ở giữa | `"105", 5` | `["1*0+5","10-5"]` | `"0"` hợp lệ, `"05"` bị loại |
| Toàn số 0 | `"00", 0` | `["0+0","0-0","0*0"]` | `"00"` bị loại dù bằng 0 |
| Toàn số 0 dài | `"000", 0` | 9 biểu thức | `3^2` cách đặt toán tử, không được nối |
| Một chữ số | `"0", 0` / `"1", 1` / `"1", 0` | `["0"]` / `["1"]` / `[]` | không có khe nào |
| Số 0 ở cuối | `"10", 10` / `"10", 1` / `"10", 0` | `["10"]` / `["1+0","1-0"]` / `["1*0"]` | số 0 cuối hợp lệ |
| Ưu tiên nhân | `"123", 9` | `["12-3"]` | không được tính `1+2*3` thành 9 |
| Target âm | `"523", -1` | `["5-2*3"]` | `prev` mang dấu âm |
| Số lớn | `"2147483647", 2147483647` | `["2147483647"]` | toán hạng 10 chữ số, target biên `2^31 - 1` |
| Nhiều đáp án | `"123456789", 45` | so với trọng tài | 121 đáp án |

**Phần 2 — test đối chiếu chéo trên input ngẫu nhiên.** Với `random.seed(282)` cố định, sinh 300 chuỗi ngắn (1–6 chữ số, **cố ý nhiều số 0** để thử luật số 0). Target được chọn một nửa bằng giá trị của một biểu thức ngẫu nhiên hợp lệ (để chắc có đáp án), một nửa là số ngẫu nhiên nhỏ (thường không có đáp án). Mọi cách đều phải cho **cùng tập đáp án** với trọng tài Cách 1.

In [23]:
def check(name: str, actual: List[str], expected: List[str]) -> None:
    assert sorted(actual) == sorted(expected), (name, actual, expected)
    print(f"PASS  {name}")

tests = [
    ("ví dụ 1", "123", 6, ["1*2*3", "1+2+3"]),
    ("ví dụ 2", "232", 8, ["2*3+2", "2+3*2"]),
    ("ví dụ 3", "3456237490", 9191, []),
    ("số 0 ở giữa", "105", 5, ["1*0+5", "10-5"]),
    ("toàn số 0", "00", 0, ["0+0", "0-0", "0*0"]),
    ("toàn số 0 dài", "000", 0, ["0+0+0", "0+0-0", "0+0*0", "0-0+0", "0-0-0",
                                 "0-0*0", "0*0+0", "0*0-0", "0*0*0"]),
    ("một chữ số 0", "0", 0, ["0"]),
    ("một chữ số khớp", "1", 1, ["1"]),
    ("một chữ số không khớp", "1", 0, []),
    ("số 0 cuối, nối", "10", 10, ["10"]),
    ("số 0 cuối, cộng/trừ", "10", 1, ["1+0", "1-0"]),
    ("số 0 cuối, nhân", "10", 0, ["1*0"]),
    ("ưu tiên nhân", "123", 9, ["12-3"]),
    ("target âm", "523", -1, ["5-2*3"]),
    ("số lớn", "2147483647", 2147483647, ["2147483647"]),
]

solver = Solution()
for name, num, target, expected in tests:
    check(name, solver.addOperators(num, target), expected)
    for _, fn in methods:
        if fn is add_operators_bruteforce and len(num) == 10:
            continue                                  # trọng tài chậm, bỏ qua chuỗi 10 chữ số
        assert same_answer(fn(num, target), expected), (fn.__name__, num, target)

big = add_operators_bruteforce("123456789", 45)
check("nhiều đáp án (so với trọng tài)", solver.addOperators("123456789", 45), big)
print("Số đáp án của '123456789', 45:", len(big))
print("Tất cả test cố định đều PASS cho cả 4 cách và Solution!")

PASS  ví dụ 1
PASS  ví dụ 2
PASS  ví dụ 3


PASS  số 0 ở giữa
PASS  toàn số 0
PASS  toàn số 0 dài
PASS  một chữ số 0
PASS  một chữ số khớp
PASS  một chữ số không khớp
PASS  số 0 cuối, nối
PASS  số 0 cuối, cộng/trừ
PASS  số 0 cuối, nhân
PASS  ưu tiên nhân
PASS  target âm
PASS  số lớn


PASS  nhiều đáp án (so với trọng tài)
Số đáp án của '123456789', 45: 121
Tất cả test cố định đều PASS cho cả 4 cách và Solution!


In [24]:
random.seed(282)

def random_valid_expression(num: str) -> str:
    # Sinh một biểu thức ngẫu nhiên HỢP LỆ từ num (để target chắc chắn có đáp án)
    while True:
        ops = [random.choice(OPS) for _ in range(len(num) - 1)]
        expr = build_expression(num, ops)
        if not has_leading_zero(expr):
            return expr

trials, non_empty = 300, 0
for _ in range(trials):
    length = random.randint(1, 6)
    num = "".join(random.choice("0012345679") for _ in range(length))   # cố ý nhiều số 0
    if random.random() < 0.5:
        target = eval(random_valid_expression(num))
    else:
        target = random.randint(-20, 50)
    expected = sorted(add_operators_bruteforce(num, target))          # trọng tài
    non_empty += len(expected) > 0
    for _, fn in methods[1:]:
        assert sorted(fn(num, target)) == expected, (fn.__name__, num, target)
    assert sorted(solver.addOperators(num, target)) == expected, ("Solution", num, target)

print(f"{trials} test ngẫu nhiên: Cách 2, 3, 4 và Solution đều khớp trọng tài Cách 1.")
print(f"Trong đó {non_empty} test có ít nhất một đáp án, {trials - non_empty} test có đáp án rỗng.")

300 test ngẫu nhiên: Cách 2, 3, 4 và Solution đều khớp trọng tài Cách 1.
Trong đó 198 test có ít nhất một đáp án, 102 test có đáp án rỗng.


## 11. Vì sao Cách 3 đúng?

Ta cần chứng minh 3 điều: **mọi biểu thức trả về đều đúng**, **không bỏ sót**, và **không trùng lặp**.

### (a) Mọi biểu thức trả về đều đúng — bất biến của `dfs`

Mỗi khi `dfs(index, path, value, prev)` được gọi (với `index > 0`), ba điều sau luôn đúng:

1. `path` là một biểu thức **hợp lệ** dùng đúng các chữ số `num[:index]`, theo đúng thứ tự, không có toán hạng nào có số 0 đứng đầu.
2. `value` = giá trị của `path` theo đúng độ ưu tiên (bằng `eval(path)`).
3. `prev` = giá trị hạng tử cuối của `path`, kèm dấu.

Chứng minh theo kiểu "nếu đúng ở cha thì đúng ở con" (quy nạp): lời gọi số đầu tiên đúng hiển nhiên; ba nhánh `+`, `-`, `*` giữ bất biến như đã lập luận ở mục 6. Toán hạng mới luôn hợp lệ vì toán hạng hỏng bị `break` chặn trước. Khi `index == n`, `path` dùng hết chữ số và `value == eval(path)`, nên lưu `path` khi `value == target` là đúng.

### (b) Không bỏ sót

Mỗi biểu thức hợp lệ được xác định **duy nhất** bởi: cách cắt `num` thành các toán hạng (đều hợp lệ) + toán tử giữa mỗi hai toán hạng liền nhau. Vòng `end` thử **mọi** độ dài toán hạng tiếp theo; lệnh `break` chỉ bỏ các toán hạng có số 0 đứng đầu (vốn không được phép). Sau toán hạng đầu tiên, cả 3 toán tử đều được thử. Vậy mọi biểu thức hợp lệ đều tương ứng với một đường đi từ gốc tới lá.

### (c) Không trùng lặp

Hai đường đi khác nhau trong cây phải khác nhau ở **lần chọn đầu tiên** mà chúng tách ra: hoặc chọn toán hạng dài khác nhau (⇒ cách cắt khác), hoặc chọn toán tử khác nhau. Cả hai trường hợp đều cho ra **chuỗi biểu thức khác nhau**. Vậy không cần `set` để khử trùng.

Cell dưới kiểm tra bất biến (a) **trực tiếp**: chèn `assert` vào mọi lời gọi `dfs`, so `value` với `eval(path)` và so `prev` với giá trị hạng tử cuối tính bằng `eval`.

In [25]:
def last_term_value(path: str) -> int:
    # Hạng tử cuối = phần sau dấu '+'/'-' cuối cùng, kèm dấu đó
    k = max(path.rfind("+"), path.rfind("-"))
    if k == -1:
        return eval(path)                   # chưa có '+'/'-': cả path là một hạng tử
    sign = 1 if path[k] == "+" else -1
    return sign * eval(path[k + 1:])

def check_invariant(num: str) -> int:
    n, checked = len(num), 0
    def dfs(index: int, path: str, value: int, prev: int) -> None:
        nonlocal checked
        if index > 0:
            assert not has_leading_zero(path), path
            assert value == eval(path), (path, value)
            assert prev == last_term_value(path), (path, prev)
            checked += 1
        if index == n:
            return
        for end in range(index, n):
            cur_str = num[index:end + 1]
            if len(cur_str) > 1 and cur_str[0] == "0":
                break
            cur = int(cur_str)
            if index == 0:
                dfs(end + 1, cur_str, cur, cur)
            else:
                dfs(end + 1, path + "+" + cur_str, value + cur, cur)
                dfs(end + 1, path + "-" + cur_str, value - cur, -cur)
                dfs(end + 1, path + "*" + cur_str, value - prev + prev * cur, prev * cur)
    dfs(0, "", 0, 0)
    return checked

print("'5-2*3+4' :", last_term_value("5-2*3+4"), "| '5-2*3' :", last_term_value("5-2*3"), "| '2*3*4' :", last_term_value("2*3*4"))
random.seed(1)
total = 0
for _ in range(40):
    num = "".join(random.choice("0123456789") for _ in range(random.randint(1, 6)))
    total += check_invariant(num)
print(f"Bất biến đúng tại {total} lời gọi dfs trên 40 chuỗi ngẫu nhiên.")

'5-2*3+4' : 4 | '5-2*3' : -6 | '2*3*4' : 24
Bất biến đúng tại 10109 lời gọi dfs trên 40 chuỗi ngẫu nhiên.


## 12. Độ phức tạp

Gọi `n = len(num)` (`n <= 10`).

### Số lá của cây gọi

Như đã nói ở (b), mỗi lá ứng với một cách chọn cho `n - 1` khe, mỗi khe 4 lựa chọn (nối, `+`, `-`, `*`). Vậy khi `num` **không có số 0**, số lá **đúng bằng** `4^(n-1)`; có số 0 thì ít hơn vì bị cắt tỉa. Còn số nút trong (lời gọi chưa phải lá)? Mỗi nút trong, trừ gốc, có **ít nhất 3 con** (toán hạng 1 chữ số với 3 toán tử), nên số nút trong không vượt quá số lá. Vậy tổng số lời gọi là `O(4^n)` — bảng ở cell dưới cho thấy tổng lời gọi chỉ khoảng **1.33 lần** số lá.

### Thời gian

Ở mỗi lời gọi, việc cắt chuỗi `num[index:end+1]`, đổi `int(...)` và tạo chuỗi `path + op + cur_str` tốn `O(n)`. Vậy:

- **Cách 3: `O(n · 4^n)`** trong trường hợp xấu nhất.
- **Cách 4:** bớt được chi phí sao chép chuỗi ở các nút không khớp, nhưng khi một lá khớp vẫn tốn `O(n)` để `join`, và cận trên vẫn viết là `O(n · 4^n)`.

Có thể làm tốt hơn hàm mũ không? Nói chung là **không**: chỉ riêng **kích thước đầu ra** đã có thể lớn theo hàm mũ. Ví dụ `num = "0000000000"`, `target = 0`: mọi cách đặt `+ - *` vào 9 khe (không được nối vì luật số 0) đều cho 0 ⇒ có `3^9 = 19683` đáp án, mỗi đáp án dài 19 ký tự. Thuật toán nào cũng phải tốn ít nhất chừng ấy công để **viết ra** đáp án.

### Bộ nhớ

- Độ sâu đệ quy tối đa là `n + 1` (mỗi mức dùng ít nhất 1 chữ số) ⇒ call stack `O(n)`.
- Mỗi mức giữ một chuỗi `path` dài `O(n)` ⇒ **`O(n^2)`** cho Cách 3 (Cách 4 chỉ `O(n)` vì dùng chung một list `path`).
- **Không tính** danh sách kết quả: ví dụ trên đã là 19683 chuỗi; tổng quát không quá `4^(n-1)` chuỗi, mỗi chuỗi dài tối đa `2n - 1` ký tự.

Với `n <= 10`, mọi con số này đều nhỏ: cỡ `262144` lá, chạy trong vài phần trăm giây với Cách 3. Cell dưới đếm số lá và số lời gọi thật để kiểm chứng.

In [26]:
def count_calls(num: str):
    n, leaves, calls = len(num), 0, 0
    def dfs(index: int) -> None:
        nonlocal leaves, calls
        calls += 1
        if index == n:
            leaves += 1
            return
        for end in range(index, n):
            if end > index and num[index] == "0":
                break
            for _ in range(1 if index == 0 else 3):
                dfs(end + 1)
    dfs(0)
    return leaves, calls

print(" n | 4^(n-1) | số lá ('1'*n) | tổng lời gọi | lời gọi / lá")
for n in range(1, 11):
    leaves, calls = count_calls("1" * n)
    print(f"{n:>2} | {4 ** (n - 1):>7} | {leaves:>13} | {calls:>12} | {calls / leaves:.2f}")

zeros = Solution().addOperators("0000000000", 0)
print("Số đáp án của '0000000000', 0:", len(zeros), "= 3^9 =", 3 ** 9, "| ví dụ:", zeros[:2])

 n | 4^(n-1) | số lá ('1'*n) | tổng lời gọi | lời gọi / lá
 1 |       1 |             1 |            2 | 2.00
 2 |       4 |             4 |            6 | 1.50
 3 |      16 |            16 |           22 | 1.38
 4 |      64 |            64 |           86 | 1.34
 5 |     256 |           256 |          342 | 1.34
 6 |    1024 |          1024 |         1366 | 1.33
 7 |    4096 |          4096 |         5462 | 1.33
 8 |   16384 |         16384 |        21846 | 1.33
 9 |   65536 |         65536 |        87382 | 1.33


10 |  262144 |        262144 |       349526 | 1.33
Số đáp án của '0000000000', 0: 19683 = 3^9 = 19683 | ví dụ: ['0+0+0+0+0+0+0+0+0+0', '0+0+0+0+0+0+0+0+0-0']


## 13. Bảng so sánh các cách

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ | Ưu điểm | Nhược điểm |
|---|---|---|---|---|---|
| 1. `product` + lọc + `eval` | sinh đủ `4^(n-1)` biểu thức, lọc số 0, `eval` | `O(n · 4^(n-1))`, hằng số lớn | `O(n)` | rất dễ viết, gần như không thể sai ⇒ **trọng tài** | sinh cả biểu thức hỏng; `eval` chậm |
| 2. Backtracking + `eval` ở lá | xây chuỗi theo cây, `break` khi số 0 đứng đầu | `O(n · 4^n)`, hằng số lớn | `O(n^2)` | không sinh biểu thức hỏng | vẫn `eval` lại từ đầu ở mỗi lá |
| 3. Backtracking + `value`/`prev` | cập nhật giá trị dần, phép nhân dùng `value - prev + prev*cur` | `O(n · 4^n)`, hằng số nhỏ | `O(n^2)` | nhanh, không `eval`, code ngắn ⇒ **cách chính** | cần hiểu kỹ `prev` và dấu của nó |
| 4. Như 3 + `path` list `append`/`pop` | không sao chép chuỗi ở mọi nút, `cur = cur*10 + d` | `O(n · 4^n)`, đo thực tế ngang Cách 3 | `O(n)` | tiết kiệm bộ nhớ nhất | dễ `pop` lệch; lợi ích tốc độ không đáng kể khi `n <= 10` |

Ghi nhớ: **cả 4 cách đều là hàm mũ** — bài này không có lời giải đa thức vì đầu ra có thể lớn theo hàm mũ. Khác biệt nằm ở **hằng số** và ở việc **không làm việc thừa** (biểu thức hỏng, `eval` lặp lại).

## 14. Các lỗi người mới hay gặp

### Lỗi 1: Quên luật số 0 đứng đầu

Code sai: bỏ hẳn dòng kiểm `len(cur_str) > 1 and cur_str[0] == "0"`.

Vì sao sai: `int("05") = 5` nên biểu thức `1*05` có "giá trị" `5` và bị trả về cho `"105", 5`. Đề cấm toán hạng `05`.

Sửa: kiểm luật số 0 **trên chuỗi**, **trước** khi `int(...)`, và `break`.

In [27]:
def wrong_no_leading_zero_check(num: str, target: int) -> List[str]:
    n, result = len(num), []
    def dfs(index, path, value, prev):
        if index == n:
            if value == target:
                result.append(path)
            return
        for end in range(index, n):
            cur_str = num[index:end + 1]           # SAI: không kiểm số 0 đứng đầu
            cur = int(cur_str)
            if index == 0:
                dfs(end + 1, cur_str, cur, cur)
            else:
                dfs(end + 1, path + "+" + cur_str, value + cur, cur)
                dfs(end + 1, path + "-" + cur_str, value - cur, -cur)
                dfs(end + 1, path + "*" + cur_str, value - prev + prev * cur, prev * cur)
    dfs(0, "", 0, 0)
    return result

print("sai :", wrong_no_leading_zero_check("105", 5))   # thừa '1*05'
print("đúng:", add_operators_backtrack("105", 5))
print("sai :", wrong_no_leading_zero_check("00", 0))    # thừa '00'

sai : ['1*0+5', '1*05', '10-5']
đúng: ['1*0+5', '10-5']
sai : ['0+0', '0-0', '0*0', '00']


### Lỗi 2: Kiểm số 0 "quá tay" — loại luôn cả toán hạng `"0"`

Code sai:

```python
if cur_str[0] == "0":
    break
```

Vì sao sai: điều kiện này chặn cả `"0"` (một chữ số), trong khi `"0"` là toán hạng **hợp lệ**. Hậu quả: `"105", 5` mất đáp án `1*0+5`; `"00", 0` trả về rỗng; `"10", 1` cũng rỗng (mất `1+0`, `1-0`).

Sửa: thêm điều kiện **dài hơn 1 chữ số**: `len(cur_str) > 1 and cur_str[0] == "0"` (hoặc `end > index and num[index] == "0"`).

In [28]:
def wrong_zero_too_strict(num: str, target: int) -> List[str]:
    n, result = len(num), []
    def dfs(index, path, value, prev):
        if index == n:
            if value == target:
                result.append(path)
            return
        for end in range(index, n):
            cur_str = num[index:end + 1]
            if cur_str[0] == "0":                  # SAI: chặn cả "0"
                break
            cur = int(cur_str)
            if index == 0:
                dfs(end + 1, cur_str, cur, cur)
            else:
                dfs(end + 1, path + "+" + cur_str, value + cur, cur)
                dfs(end + 1, path + "-" + cur_str, value - cur, -cur)
                dfs(end + 1, path + "*" + cur_str, value - prev + prev * cur, prev * cur)
    dfs(0, "", 0, 0)
    return result

for num, target in [("105", 5), ("00", 0), ("10", 1)]:
    print(f"{num!r:<6} {target}: sai = {wrong_zero_too_strict(num, target)!s:<10} đúng = {add_operators_backtrack(num, target)}")

'105'  5: sai = ['10-5']   đúng = ['1*0+5', '10-5']
'00'   0: sai = []         đúng = ['0+0', '0-0', '0*0']
'10'   1: sai = []         đúng = ['1+0', '1-0']


### Lỗi 3: Quên dấu của `prev` khi trừ

Code sai: ở nhánh `-` truyền `prev = cur` thay vì `prev = -cur`.

Vì sao sai: hạng tử cuối của `5-2` là `-2`. Nếu lưu `prev = 2` thì khi gặp `*3`, công thức tính `3 - 2 + 2*3 = 7` thay vì `-1` (xem bảng ở mục 6). Hậu quả: `"523", -1` mất đáp án `5-2*3`, còn `"523", 7` lại trả thừa `5-2*3`.

Sửa: `dfs(end + 1, path + "-" + cur_str, value - cur, -cur)`.

In [29]:
def wrong_prev_sign(num: str, target: int) -> List[str]:
    n, result = len(num), []
    def dfs(index, path, value, prev):
        if index == n:
            if value == target:
                result.append(path)
            return
        for end in range(index, n):
            cur_str = num[index:end + 1]
            if len(cur_str) > 1 and cur_str[0] == "0":
                break
            cur = int(cur_str)
            if index == 0:
                dfs(end + 1, cur_str, cur, cur)
            else:
                dfs(end + 1, path + "+" + cur_str, value + cur, cur)
                dfs(end + 1, path + "-" + cur_str, value - cur, cur)     # SAI: phải là -cur
                dfs(end + 1, path + "*" + cur_str, value - prev + prev * cur, prev * cur)
    dfs(0, "", 0, 0)
    return result

for target in [-1, 7]:
    print(f"'523', {target:>2}: sai = {wrong_prev_sign('523', target)!s:<20} đúng = {add_operators_backtrack('523', target)}")

'523', -1: sai = []                   đúng = ['5-2*3']
'523',  7: sai = ['5-2*3', '5*2-3']   đúng = ['5*2-3']


### Lỗi 4: Không tách riêng số đầu tiên

Code sai: bỏ nhánh `if index == 0`, luôn thử 3 toán tử, kể cả trước chữ số đầu tiên (với `path = ""`, `value = 0`, `prev = 0`).

Vì sao sai: sinh ra biểu thức bắt đầu bằng toán tử như `+1+2+3` hoặc `*1+2*3` — không hợp lệ vì toán tử là **hai ngôi**. Tệ hơn, `*1` với `value = 0`, `prev = 0` cho `0 - 0 + 0*1 = 0`, tức là "nuốt mất" số đầu, nên `*1+2*3` lại được tính ra 6 và bị trả về cho `"123", 6`.

Sửa: số đầu tiên chỉ có một lựa chọn: `dfs(end + 1, cur_str, cur, cur)`.

In [30]:
def wrong_no_first_case(num: str, target: int) -> List[str]:
    n, result = len(num), []
    def dfs(index, path, value, prev):
        if index == n:
            if value == target:
                result.append(path)
            return
        for end in range(index, n):
            cur_str = num[index:end + 1]
            if len(cur_str) > 1 and cur_str[0] == "0":
                break
            cur = int(cur_str)                     # SAI: không có nhánh index == 0
            dfs(end + 1, path + "+" + cur_str, value + cur, cur)
            dfs(end + 1, path + "-" + cur_str, value - cur, -cur)
            dfs(end + 1, path + "*" + cur_str, value - prev + prev * cur, prev * cur)
    dfs(0, "", 0, 0)
    return result

print("sai :", wrong_no_first_case("123", 6))
print("đúng:", add_operators_backtrack("123", 6))

sai : ['+1+2+3', '+1*2*3', '*1+2*3']
đúng: ['1+2+3', '1*2*3']


### Các lỗi khác cần nhớ

**Lỗi 5 — Phép nhân cập nhật `value * cur`.** Đã thấy ở mục 5: bỏ qua độ ưu tiên, tính `1+2*3` thành `(1+2)*3 = 9`. Luôn dùng `value - prev + prev * cur`.

**Lỗi 6 — Đổi sang `int` trước rồi mới kiểm số 0.** Viết `cur = int(cur_str)` rồi kiểm `cur` thì không còn biết `"05"` hay `"5"` (cả hai đều là `5`). Phải kiểm trên **chuỗi** (hoặc dùng `end > index and num[index] == "0"`).

**Lỗi 7 — Dùng `continue` thay cho `break`.** Vẫn **đúng** (vì mọi toán hạng dài hơn đều bị loại), nhưng phí công cắt chuỗi và kiểm thêm vô ích. `break` thể hiện đúng hiểu biết: "đã bắt đầu bằng `'0'` thì dài thêm cũng vô ích".

**Lỗi 8 — Tràn số ở Java/C++.** Python có số nguyên không giới hạn nên không lo. Nhưng ở Java/C++, toán hạng `9999999999` (10 chữ số) đã vượt `int` 32-bit, và `prev * cur` còn lớn hơn nữa ⇒ phải dùng `long`. Đây là lý do nhiều lời giải Java khai báo `value` và `prev` kiểu `long`.

**Lỗi 9 — So sánh kết quả không sắp xếp.** `["1+2+3", "1*2*3"] == ["1*2*3", "1+2+3"]` là `False`. Đề chấp nhận mọi thứ tự, nên khi test phải so `sorted(...)`.

**Lỗi 10 — Tin vào `eval` để xử lý số 0.** `eval("1*05")` báo `SyntaxError` (chương trình dừng), còn `eval("1+00")` lại chạy êm. Nếu dùng `eval` (Cách 1, 2) thì phải lọc số 0 **trước**.

## 15. Mẹo nhận dạng pattern

- **"Trả về TẤT CẢ" + input rất nhỏ (`n <= 10`)** ⇒ gần như chắc chắn là **backtracking / liệt kê**. Đầu ra có thể lớn theo hàm mũ thì không có cách nào tránh được hàm mũ.
- **"Chèn thứ gì đó vào giữa các ký tự" + được "nối" ký tự** ⇒ vòng `for end in range(index, n)` để chọn **mảnh tiếp theo** `s[index:end + 1]`, rồi đệ quy với `end + 1`.
- **Có phép nhân (hoặc toán tử ưu tiên cao hơn) cần tính dần** ⇒ giữ **hạng tử cuối** (`prev` / `last`) để có thể "gỡ ra rồi làm lại". Cùng một ý tưởng xuất hiện trong bài tính biểu thức (Basic Calculator II).
- **"Không có số 0 đứng đầu"** ⇒ `if end > index and s[index] == "0": break`. Câu này lặp lại y hệt ở các bài cắt chuỗi số.
- **Thứ tự đáp án tùy ý** ⇒ test bằng `sorted`, và luôn có một **trọng tài** brute force cho input nhỏ.

## 16. Biến thể và bài liên quan

| Bài | Liên hệ với bài này |
|---|---|
| LeetCode 494 — Target Sum | gán `+`/`-` trước **mỗi số** (không nối, không nhân); chỉ **đếm** số cách ⇒ giải được bằng DP/memo |
| LeetCode 227 — Basic Calculator II | tính giá trị chuỗi có `+ - * /` đúng ưu tiên — dùng đúng mẹo **hạng tử cuối** (xem Bài tập 5) |
| LeetCode 224 — Basic Calculator (Hard) | tính biểu thức có dấu ngoặc — dùng stack |
| LeetCode 241 — Different Ways to Add Parentheses | toán tử đã có sẵn, ta chọn cách **đặt ngoặc** ⇒ chia để trị |
| LeetCode 93 — Restore IP Addresses | cắt chuỗi số thành 4 phần, cùng luật **không có số 0 đứng đầu** |
| LeetCode 306 — Additive Number | cắt chuỗi số thành dãy cộng dồn, cùng luật số 0 đứng đầu |
| LeetCode 679 — 24 Game (Hard) | thử mọi phép toán giữa các số để đạt 24 — backtracking trên biểu thức |
| LeetCode 17 — Letter Combinations of a Phone Number | backtracking cơ bản: mỗi vị trí một tập lựa chọn |

**Biến thể tự nghĩ thêm:**

- Chỉ cho phép `+` và `-` (Bài tập 4) ⇒ không cần `prev`.
- Thêm phép chia `/`: mẹo `prev` vẫn dùng được (`value - prev + prev / cur`, cẩn thận chia cho 0 và quy ước làm tròn), nhưng đề phải nói rõ phép chia nguyên hay chia thực.
- Tìm biểu thức dùng **ít toán tử nhất** (Bài tập 6) ⇒ backtracking + cắt nhánh khi đã dùng nhiều toán tử hơn kỷ lục.

Cell dưới là một thí nghiệm nhỏ giúp "cảm" bài toán: với `num = "1234"` (64 biểu thức), mỗi giá trị đạt được bởi bao nhiêu biểu thức? Ta dùng `Counter` (từ `collections`) — một `dict` đặc biệt dùng để **đếm**: `Counter([...])[x]` là số lần `x` xuất hiện.

In [31]:
from collections import Counter

def all_values(num: str) -> Counter:
    # Đếm giá trị của MỌI biểu thức hợp lệ (dùng lại ý tưởng value/prev, không lọc target)
    n, values = len(num), Counter()
    def dfs(index, value, prev):
        if index == n:
            values[value] += 1
            return
        for end in range(index, n):
            if end > index and num[index] == "0":
                break
            cur = int(num[index:end + 1])
            if index == 0:
                dfs(end + 1, cur, cur)
            else:
                dfs(end + 1, value + cur, cur)
                dfs(end + 1, value - cur, -cur)
                dfs(end + 1, value - prev + prev * cur, prev * cur)
    dfs(0, 0, 0)
    return values

vals = all_values("1234")
print("tổng số biểu thức:", sum(vals.values()), "| số giá trị khác nhau:", len(vals))
print("5 giá trị hay gặp nhất:", vals.most_common(5))
print("target 10 có", vals[10], "biểu thức:", add_operators_backtrack("1234", 10))
print("target 11 có", vals[11], "biểu thức")
only_one = sum(1 for v in vals.values() if v == 1)
print(f"{only_one}/{len(vals)} giá trị chỉ đạt được bởi ĐÚNG 1 biểu thức -> số đáp án thường rất ít so với 4^(n-1)")

tổng số biểu thức: 64 | số giá trị khác nhau: 55
5 giá trị hay gặp nhất: [(11, 3), (10, 2), (2, 2), (-9, 2), (3, 2)]
target 10 có 2 biểu thức: ['1+2+3+4', '1*2*3+4']
target 11 có 3 biểu thức
47/55 giá trị chỉ đạt được bởi ĐÚNG 1 biểu thức -> số đáp án thường rất ít so với 4^(n-1)


## 17. Bài tập vận dụng (từ dễ đến khó)

Làm theo thứ tự. Bài 1–2 làm trên giấy; bài 3–6 viết code trong các cell TODO (các cell này **chỉ định nghĩa hàm**, test để sẵn ở dạng comment — làm xong thì bỏ comment để tự kiểm). Gợi ý ở mục 18, đáp án ở mục 19 cuối notebook.

### Bài 1 — Đoán output (không chạy code)

a) Toán hạng `"0"`, `"00"`, `"101"`, `"010"` cái nào hợp lệ?

b) `Solution().addOperators("10", 1)` trả về gì?

c) Với `num = "1234"`, Cách 1 sinh ra bao nhiêu bộ `ops`? Có bộ nào bị loại vì số 0 đứng đầu không?

d) Kết quả của `addOperators("00", 0)` có chứa `"00"` không? Vì sao?

### Bài 2 — Tính tay `value` và `prev`

Lập bảng `value`, `prev` sau từng bước khi xây biểu thức `2*3-4*5+6` theo quy tắc ở mục 6 (bắt đầu từ số đầu `2`). Giá trị cuối cùng là bao nhiêu? Kiểm lại bằng cách tính tổng các hạng tử có dấu.

### Bài 3 — Điền chỗ trống: hoàn thiện Cách 3

Điền các `TODO` trong hàm `exercise_add_operators`. Khung đệ quy đã có sẵn; bạn cần viết:

1. base case;
2. điều kiện số 0 đứng đầu;
3. nhánh số đầu tiên;
4. ba nhánh `+`, `-`, `*` với `value`, `prev` đúng.

In [32]:
def exercise_add_operators(num: str, target: int) -> List[str]:
    n = len(num)
    result = []

    def dfs(index: int, path: str, value: int, prev: int) -> None:
        # TODO 1: base case — nếu index == n thì lưu path khi value == target, rồi return
        for end in range(index, n):
            cur_str = num[index:end + 1]
            # TODO 2: nếu cur_str có số 0 đứng đầu thì break
            cur = int(cur_str)
            # TODO 3: nếu index == 0: gọi dfs cho số đầu tiên (không có toán tử)
            # TODO 4: ngược lại: gọi dfs cho 3 nhánh '+', '-', '*'
            pass

    dfs(0, "", 0, 0)
    return result

# Tự làm xong rồi mới bỏ comment để test:
# assert sorted(exercise_add_operators("123", 6)) == ["1*2*3", "1+2+3"]
# assert sorted(exercise_add_operators("105", 5)) == ["1*0+5", "10-5"]
# assert sorted(exercise_add_operators("00", 0)) == ["0*0", "0+0", "0-0"]
# assert exercise_add_operators("523", -1) == ["5-2*3"]

### Bài 4 — Biến thể: chỉ được chèn `+` và `-`

Viết `exercise_add_sub_only(num, target)`: giống đề gốc nhưng **không có phép nhân** (vẫn được nối chữ số, vẫn giữ luật số 0 đứng đầu). Trả về mọi biểu thức bằng `target`.

Câu hỏi suy nghĩ: còn cần biến `prev` không?

Ví dụ: `"123", 24 -> ["1+23"]`; `"105", 5 -> ["10-5"]`; câu đố nổi tiếng `"123456789", 100` (chỉ dùng `+`, `-`) có đúng **11** đáp án, ví dụ `"123-45-67+89"`.

In [33]:
def exercise_add_sub_only(num: str, target: int) -> List[str]:
    # TODO: backtracking giống Cách 3 nhưng chỉ có nhánh '+' và '-'
    pass

# Tự làm xong rồi mới bỏ comment để test:
# assert exercise_add_sub_only("123", 6) == ["1+2+3"]
# assert exercise_add_sub_only("123", 24) == ["1+23"]
# assert exercise_add_sub_only("105", 5) == ["10-5"]
# assert sorted(exercise_add_sub_only("00", 0)) == ["0+0", "0-0"]
# assert len(exercise_add_sub_only("123456789", 100)) == 11

### Bài 5 — Vận dụng mẹo `prev`: tính giá trị biểu thức **không dùng `eval`**

Viết `exercise_evaluate(expr)` nhận chuỗi chỉ gồm chữ số và `+`, `-`, `*` (không có dấu cách, không có ngoặc, không có dấu trừ đứng đầu) và trả về giá trị đúng ưu tiên. **Không** dùng `eval`.

Gợi ý chung: duyệt từng ký tự, gom chữ số thành số bằng `cur * 10 + d`; mỗi khi gặp toán tử (hoặc hết chuỗi) thì áp dụng **toán tử đứng trước số vừa gom** với quy tắc `value`/`prev`. Đây chính là cốt lõi của LeetCode 227 — Basic Calculator II.

In [34]:
def exercise_evaluate(expr: str) -> int:
    # TODO: duyệt ký tự, gom số, áp dụng quy tắc value/prev cho toán tử đứng trước số
    pass

# Tự làm xong rồi mới bỏ comment để test:
# assert exercise_evaluate("1+2*3*4") == 25
# assert exercise_evaluate("5-2*3") == -1
# assert exercise_evaluate("12*3-4*5+6") == 22
# assert exercise_evaluate("7") == 7
# assert exercise_evaluate("10-5") == 5

### Bài 6 — Mini challenge: dùng **ít toán tử nhất**

Viết `exercise_min_operators(num, target)` trả về **số toán tử ít nhất** cần chèn (`+`, `-`, `*`, luật như đề gốc) để biểu thức bằng `target`; nếu không có cách nào thì trả `-1`.

Ví dụ:

- `"123", 123 -> 0` (không chèn gì)
- `"123", 24 -> 1` (`1+23`)
- `"123", 6 -> 2` (`1+2+3` hoặc `1*2*3`)
- `"3456237490", 9191 -> -1`

Thử thách thêm: **cắt nhánh** — nếu số toán tử đã dùng **không nhỏ hơn** kỷ lục tốt nhất đã tìm được thì quay lui ngay.

In [35]:
def exercise_min_operators(num: str, target: int) -> int:
    # TODO: backtracking giữ thêm số toán tử đã dùng; cập nhật kỷ lục ở lá; cắt nhánh
    pass

# Tự làm xong rồi mới bỏ comment để test:
# assert exercise_min_operators("123", 123) == 0
# assert exercise_min_operators("123", 24) == 1
# assert exercise_min_operators("123", 6) == 2
# assert exercise_min_operators("105", 5) == 1
# assert exercise_min_operators("3456237490", 9191) == -1

## 18. Gợi ý theo mức

**Bài 1**

- Mức 1: nhớ điều kiện "dài hơn 1 chữ số **và** bắt đầu bằng `0`".
- Mức 2: `"10"` có 1 khe với 4 lựa chọn: `10`, `1+0`, `1-0`, `1*0`. Tính từng cái.
- Mức 3: `"1234"` có 3 khe; không có chữ số 0 nào thì có toán hạng nào bắt đầu bằng `0` được không?

**Bài 2**

- Mức 1: bắt đầu `value = prev = 2`.
- Mức 2: gặp `*` thì `value = value - prev + prev*cur`, `prev = prev*cur`; gặp `-` thì `prev = -cur`.
- Mức 3: các hạng tử có dấu là `+(2*3)`, `-(4*5)`, `+6`.

**Bài 3**

- Mức 1: chép lại đúng bảng quy tắc ở mục 6.
- Mức 2: base case gồm 3 bước: `if index == n:` → nếu `value == target` thì `result.append(path)` → `return`. Số 0 đứng đầu: `len(cur_str) > 1 and cur_str[0] == "0"` → `break`. Số đầu tiên: `dfs(end + 1, cur_str, cur, cur)`.
- Mức 3: nhánh nhân: `dfs(end + 1, path + "*" + cur_str, value - prev + prev * cur, prev * cur)`.

**Bài 4**

- Mức 1: không có phép nhân thì không bao giờ phải "gỡ" hạng tử cuối.
- Mức 2: `dfs(index, path, value)` là đủ, bỏ hẳn `prev`.
- Mức 3: hai nhánh `dfs(end + 1, path + "+" + cur_str, value + cur)` và `dfs(end + 1, path + "-" + cur_str, value - cur)`.

**Bài 5**

- Mức 1: cần 4 biến: `value`, `prev`, `cur` (số đang gom), `op` (toán tử đứng trước `cur`, ban đầu coi như `'+'`).
- Mức 2: nếu coi số đầu tiên có toán tử `'+'` đứng trước và bắt đầu `value = prev = 0`, thì `0 + cur` cho đúng `value = prev = cur` — khỏi xử lý riêng số đầu.
- Mức 3: duyệt `for ch in expr + "+"` (thêm một toán tử "giả" ở cuối) để số cuối cùng cũng được áp dụng.

**Bài 6**

- Mức 1: thêm tham số `ops_used` vào `dfs`; số đầu tiên không tốn toán tử, mỗi nhánh `+ - *` tốn 1.
- Mức 2: giữ kỷ lục `best` (ban đầu là vô cực `float("inf")`), cập nhật ở lá khi `value == target`.
- Mức 3: đầu hàm `dfs` thêm `if ops_used >= best: return` — nhánh này không thể cho kết quả tốt hơn.

## 19. Đáp án tham khảo

Hãy tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

a) `"0"` ✔ hợp lệ (một chữ số); `"00"` ✘ (dài 2, bắt đầu bằng 0); `"101"` ✔ (bắt đầu bằng 1); `"010"` ✘.

b) Bốn biểu thức: `10 = 10`, `1+0 = 1`, `1-0 = 1`, `1*0 = 0` ⇒ kết quả `["1+0", "1-0"]` (thứ tự bất kỳ).

c) 3 khe ⇒ `4^3 = 64` bộ `ops`. Không có bộ nào bị loại, vì không có chữ số 0 nào nên không toán hạng nào bắt đầu bằng `0`.

d) **Không**. `"00"` có giá trị bằng 0 nhưng là toán hạng có số 0 đứng đầu ⇒ không hợp lệ. Kết quả là `["0+0", "0-0", "0*0"]`.

### Đáp án Bài 2

| Bước | Thêm | Tính `value` | `value` | `prev` |
|---:|---|---|---:|---:|
| 1 | `2` (số đầu) | `2` | 2 | 2 |
| 2 | `*3` | `2 - 2 + 2*3` | 6 | 6 |
| 3 | `-4` | `6 - 4` | 2 | -4 |
| 4 | `*5` | `2 - (-4) + (-4)*5 = 2 + 4 - 20` | -14 | -20 |
| 5 | `+6` | `-14 + 6` | -8 | 6 |

Kiểm lại bằng hạng tử: `+(2*3) - (4*5) + 6 = 6 - 20 + 6 = -8` ✔.

In [36]:
# Kiểm chứng Bài 1
print("a)", [(s, is_valid_operand(s)) for s in ["0", "00", "101", "010"]])
print("b)", Solution().addOperators("10", 1))
print("c)", len(list(product(OPS, repeat=3))), "bộ ops; bị loại:",
      sum(has_leading_zero(build_expression("1234", ops)) for ops in product(OPS, repeat=3)))
print("d)", Solution().addOperators("00", 0), "| có '00'?", "00" in Solution().addOperators("00", 0))

# Kiểm chứng Bài 2
print()
assert simulate("2*3-4*5+6") == eval("2*3-4*5+6") == -8
print("Bài 2 PASS")

a) [('0', True), ('00', False), ('101', True), ('010', False)]
b) ['1+0', '1-0']
c) 64 bộ ops; bị loại: 0
d) ['0+0', '0-0', '0*0'] | có '00'? False

số đầu 2   -> value = 2    prev = 2
thêm *3    -> value = 6    prev = 6    (tính: 2 - (2) + (2)*3)
thêm -4    -> value = 2    prev = -4   (tính: 6 - 4)
thêm *5    -> value = -14  prev = -20  (tính: 2 - (-4) + (-4)*5)
thêm +6    -> value = -8   prev = 6    (tính: -14 + 6)
Bài 2 PASS


In [37]:
# ===== Đáp án Bài 3 =====
def answer_add_operators(num: str, target: int) -> List[str]:
    n = len(num)
    result = []

    def dfs(index: int, path: str, value: int, prev: int) -> None:
        if index == n:                                       # TODO 1
            if value == target:
                result.append(path)
            return
        for end in range(index, n):
            cur_str = num[index:end + 1]
            if len(cur_str) > 1 and cur_str[0] == "0":       # TODO 2
                break
            cur = int(cur_str)
            if index == 0:                                   # TODO 3
                dfs(end + 1, cur_str, cur, cur)
            else:                                            # TODO 4
                dfs(end + 1, path + "+" + cur_str, value + cur, cur)
                dfs(end + 1, path + "-" + cur_str, value - cur, -cur)
                dfs(end + 1, path + "*" + cur_str, value - prev + prev * cur, prev * cur)

    dfs(0, "", 0, 0)
    return result

assert sorted(answer_add_operators("123", 6)) == ["1*2*3", "1+2+3"]
assert sorted(answer_add_operators("105", 5)) == ["1*0+5", "10-5"]
assert sorted(answer_add_operators("00", 0)) == ["0*0", "0+0", "0-0"]
assert answer_add_operators("523", -1) == ["5-2*3"]
print("Bài 3 PASS")


# ===== Đáp án Bài 4 =====
def answer_add_sub_only(num: str, target: int) -> List[str]:
    n = len(num)
    result = []

    def dfs(index: int, path: str, value: int) -> None:     # không cần prev
        if index == n:
            if value == target:
                result.append(path)
            return
        for end in range(index, n):
            if end > index and num[index] == "0":
                break
            cur_str = num[index:end + 1]
            cur = int(cur_str)
            if index == 0:
                dfs(end + 1, cur_str, cur)
            else:
                dfs(end + 1, path + "+" + cur_str, value + cur)
                dfs(end + 1, path + "-" + cur_str, value - cur)

    dfs(0, "", 0)
    return result

assert answer_add_sub_only("123", 6) == ["1+2+3"]
assert answer_add_sub_only("123", 24) == ["1+23"]
assert answer_add_sub_only("105", 5) == ["10-5"]
assert sorted(answer_add_sub_only("00", 0)) == ["0+0", "0-0"]
hundred = answer_add_sub_only("123456789", 100)
assert len(hundred) == 11
# đối chiếu: đúng bằng các đáp án của Cách 3 không chứa dấu '*'
assert sorted(hundred) == sorted(e for e in add_operators_backtrack("123456789", 100) if "*" not in e)
print("Bài 4 PASS | 3 trong 11 đáp án của 100:", hundred[-3:])

Bài 3 PASS
Bài 4 PASS | 3 trong 11 đáp án của 100: ['123-4-5-6-7+8-9', '123+45-67+8-9', '123-45-67+89']


In [38]:
# ===== Đáp án Bài 5 =====
def answer_evaluate(expr: str) -> int:
    value, prev, cur, op = 0, 0, 0, "+"      # coi như có '+' trước số đầu tiên
    for ch in expr + "+":                    # '+' giả ở cuối để xử lý số cuối
        if ch.isdigit():
            cur = cur * 10 + int(ch)         # gom chữ số thành số
            continue
        if op == "+":
            value, prev = value + cur, cur
        elif op == "-":
            value, prev = value - cur, -cur
        else:  # op == "*"
            value, prev = value - prev + prev * cur, prev * cur
        op, cur = ch, 0                      # toán tử mới, bắt đầu gom số mới
    return value

for e in ["1+2*3*4", "5-2*3", "12*3-4*5+6", "7", "10-5"]:
    assert answer_evaluate(e) == eval(e), e
random.seed(5)
for _ in range(500):
    digits = "".join(random.choice("123456789") for _ in range(random.randint(1, 7)))
    e = build_expression(digits, [random.choice(OPS) for _ in range(len(digits) - 1)])
    assert answer_evaluate(e) == eval(e), e
print("Bài 5 PASS (5 ví dụ + 500 biểu thức ngẫu nhiên khớp eval)")


# ===== Đáp án Bài 6 =====
def answer_min_operators(num: str, target: int) -> int:
    n = len(num)
    best = float("inf")

    def dfs(index: int, value: int, prev: int, ops_used: int) -> None:
        nonlocal best
        if ops_used >= best:                 # cắt nhánh: không thể tốt hơn kỷ lục
            return
        if index == n:
            if value == target:
                best = ops_used
            return
        for end in range(index, n):
            if end > index and num[index] == "0":
                break
            cur = int(num[index:end + 1])
            if index == 0:
                dfs(end + 1, cur, cur, 0)
            else:
                dfs(end + 1, value + cur, cur, ops_used + 1)
                dfs(end + 1, value - cur, -cur, ops_used + 1)
                dfs(end + 1, value - prev + prev * cur, prev * cur, ops_used + 1)

    dfs(0, 0, 0, 0)
    return -1 if best == float("inf") else best

assert answer_min_operators("123", 123) == 0
assert answer_min_operators("123", 24) == 1
assert answer_min_operators("123", 6) == 2
assert answer_min_operators("105", 5) == 1
assert answer_min_operators("3456237490", 9191) == -1

# Đối chiếu với cách "liệt kê hết rồi lấy min" trên input ngẫu nhiên
def count_ops(expr: str) -> int:
    return sum(ch in "+-*" for ch in expr)

random.seed(6)
for _ in range(200):
    num = "".join(random.choice("0123456789") for _ in range(random.randint(1, 6)))
    target = random.randint(-10, 30)
    found = add_operators_backtrack(num, target)
    expected = min(map(count_ops, found)) if found else -1
    assert answer_min_operators(num, target) == expected, (num, target)
print("Bài 6 PASS | '123456789', 100 cần ít nhất", answer_min_operators("123456789", 100), "toán tử")

Bài 5 PASS (5 ví dụ + 500 biểu thức ngẫu nhiên khớp eval)


Bài 6 PASS | '123456789', 100 cần ít nhất 3 toán tử


## 20. Template ngắn nên nhớ

```python
def add_operators(num, target):
    n, result = len(num), []

    def dfs(index, path, value, prev):          # prev = hạng tử cuối, có dấu
        if index == n:
            if value == target:
                result.append(path)
            return
        for end in range(index, n):
            if end > index and num[index] == "0":   # "05" -> dừng hẳn
                break
            s = num[index:end + 1]
            cur = int(s)
            if index == 0:                          # số đầu: không có toán tử
                dfs(end + 1, s, cur, cur)
            else:
                dfs(end + 1, path + "+" + s, value + cur, cur)
                dfs(end + 1, path + "-" + s, value - cur, -cur)
                dfs(end + 1, path + "*" + s, value - prev + prev * cur, prev * cur)

    dfs(0, "", 0, 0)
    return result
```

Câu thần chú: **"Nhân thì gỡ hạng tử cuối ra rồi lắp hạng tử mới vào: `value - prev + prev * cur`."**

## 21. Checklist tự kiểm tra trước khi submit

- [ ] Vòng `end` chạy từ `index` đến `n - 1` (nối chữ số = chọn toán hạng dài hơn).
- [ ] Luật số 0: `end > index and num[index] == "0"` ⇒ `break`; **vẫn cho phép** toán hạng `"0"`.
- [ ] Kiểm số 0 trên **chuỗi**, trước khi `int(...)`.
- [ ] Số đầu tiên (`index == 0`) **không** có toán tử đứng trước, `value = prev = cur`.
- [ ] Nhánh `-` truyền `prev = -cur` (có dấu).
- [ ] Nhánh `*` dùng `value - prev + prev * cur` và `prev * cur`.
- [ ] Base case `index == n` so `value == target` rồi `return`.
- [ ] Không dùng `eval` trong lời giải nộp (chỉ dùng làm trọng tài khi tự test).
- [ ] Test: 3 ví dụ đề, `"105", 5`, `"00", 0`, một chữ số, target âm; so bằng `sorted`.
- [ ] Nếu viết bằng Java/C++: `value`, `prev`, `cur` kiểu `long`.